# Qiskit Fall Fest 2026, Session 5: Real hardware, noise, and error correction

Fri, Nov 20, 2026, 5:00 to 6:00 PM ET, on Zoom

Covers: $T_1$ and $T_2$, transpilation, error mitigation, and the road to the surface code.

This is the last of five sessions this week. Tonight has three parts.

- Seminar: what goes wrong in a real machine, and what can be done.
- Lab: read calibration data, transpile, run with and without mitigation.
- Challenge 5 opens at 6:00 PM. All five close Sunday at 11:59 PM.

Until now a quantum processor was a box: a circuit went in and results came out. Tonight you open the box.

The words in the lines above, in one sentence each. A qubit is the quantum version of a bit, and a quantum processor is the chip that holds the qubits. A gate is one operation applied to one or two qubits, and a circuit is a list of gates, usually ending in a measurement. Noise is everything that makes the results of a real processor differ from the ideal ones: gates and measurements that go slightly wrong, and qubits that lose their state while they wait. $T_1$ and $T_2$ are two times that say how long a qubit keeps what was stored in it. Transpilation is the rewriting of a circuit so that it fits one particular processor. Error mitigation is a family of methods that bring the results of a noisy processor closer to the ideal ones, by running more circuits and correcting the results afterwards. Error correction goes further: it spreads the information of one qubit over several, so that an error can be found and repaired while the circuit is still running. The surface code is the error-correcting code that the last part of the seminar leads up to. Calibration data is the record of how well each qubit and each gate of a processor works. Qiskit is the open-source Python library, developed by IBM, that you use all week to write quantum programs and run them.

This notebook stands on its own. If you missed the earlier sessions, start here anyway: Part 1 opens with the few ideas from earlier in the week that tonight needs, and every other term is explained where it first appears.

## What you will be able to do

- Read a processor's calibration data: $T_1$, $T_2$, readout error and two-qubit gate error. Readout error is the chance that a measurement reports the wrong bit.
- Explain why a circuit is transpiled, and count what routing costs. Routing means moving the states of qubits across the chip until two qubits that must interact sit next to each other.
- Transpile one circuit several ways and keep the version with the fewest two-qubit gates.
- Run a circuit raw and with error mitigation, and compare both results with the ideal value.
- Explain how the three-qubit bit-flip code finds and fixes an error without reading the data. The bit-flip code stores one qubit in three, so that a single flipped qubit can be found and repaired.

## How to use this notebook

- Run the cells in order, from the top. A cell is one box of text or code. To run a code cell, click it and press Shift+Enter.
- Part 1 follows tonight's slides. Each idea has a short recap, and often a small cell you can run to see the idea at work.
- Part 2 is the lab. The code under "Try it" and "Lab" is complete. Run it, read the output, then change it and run it again.
- After each lab there is a "Your turn" exercise. Replace the line `# YOUR CODE HERE` with your own code, run the cell, then run the check cell below it. A check that fails stops with a red error message. Its last line is the hint. Run all (Colab: Runtime menu; Jupyter: Run menu) stops at the first check whose exercise you have not done yet. Do the exercise there, or run the cells after it one by one.
- Everything runs on a simulator, a program that imitates a quantum computer, on the computer that runs this notebook (your laptop, or the server of Google Colab or qBraid Lab, two websites that run notebooks in your browser), unless you change the switches in the Setup section.
- Use this notebook during the session and again afterwards on your own.

## Setup

Run the next three code cells in order, every time you open this notebook.

**Where your circuits run.** The first cell holds three switches.

- `USE_HARDWARE = False` keeps everything on the computer that runs this notebook (your laptop, or the server of Google Colab or qBraid Lab, two websites that run notebooks in your browser). Circuits run on a simulator, a program that imitates a quantum computer. No account is needed. Use this setting during the lab.
- `USE_HARDWARE = True` with `HARDWARE = "ibm"` sends the hardware cells to a real IBM quantum processor through your own IBM Quantum account. A hardware cell is a cell that can send work to a real quantum computer, and a processor is the chip that holds its qubits. This is the first choice for real hardware.
- `USE_HARDWARE = True` with `HARDWARE = "qbraid"` is the second option. qBraid is a service that gives access to quantum computers from several makers. The cells marked "second option" then use your qBraid account to reach the other quantum computers qBraid offers. qBraid calls each of them a device, and `QBRAID_DEVICE` holds the id of the device you pick there.

After you change a switch, run the switches cell below and the setup cell after it again.

In [ ]:
USE_HARDWARE = False      # True sends jobs to a real quantum computer
HARDWARE = "ibm"          # "ibm" (first choice) or "qbraid" (second option)
QBRAID_DEVICE = "PASTE_DEVICE_ID"

The next cell checks that the tested versions of the packages are installed. If one is missing or has a different version, the cell installs it and then stops with one line that asks you to restart the kernel. The kernel is the Python process that runs the cells of this notebook. After the restart, run the notebook again from the top.

The cell also defines `connect()`, the one helper that every hardware cell calls. `connect("ibm")` gives the lab code a service, the object you ask for a backend. A backend is the machine, real or simulated, that runs a circuit. With `USE_HARDWARE = False` the service hands out a simulated copy of the IBM processor `ibm_fez`, built from a saved record of that machine's error rates. Qiskit names this copy `fake_fez`.

In [ ]:
# Setup. Run this cell every time, right after the switches cell above. It checks the packages, then defines connect().
import sys
from importlib.metadata import PackageNotFoundError, version

PINNED = {"qiskit": "2.6.2", "qiskit-ibm-runtime": "0.52.1", "qiskit-aer": "0.18.2"}
if str(HARDWARE).strip().lower() == "qbraid":
    PINNED["qbraid"] = "0.13.0"                        # needed only for the second option
EXTRA = ["matplotlib", "pylatexenc", "scikit-learn"]   # any version will do


def _found(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return None


def _missing():
    wanted = [f"{name}=={pin}" for name, pin in PINNED.items() if _found(name) != pin]
    return wanted + [name for name in EXTRA if _found(name) is None]


class StopHere(Exception):
    """Stops the run with one plain line in place of a long error report."""

    def _render_traceback_(self):
        return [str(self)]


if sys.version_info < (3, 11):
    raise StopHere("This notebook needs Python 3.11 or newer. This kernel runs Python "
                   + sys.version.split()[0] + ". Choose a newer Python, then run this cell again.")

if _missing():
    _spec = " ".join(_missing())
    %pip install -q {_spec}
    if _missing():
        raise StopHere("Some packages could not be installed: " + ", ".join(_missing())
                       + ". Read the message above this line, or ask an organizer.")
    raise StopHere("Packages were installed. Restart the kernel now, then run the notebook "
                   "again from the top. (Jupyter and qBraid Lab: Kernel menu, Restart Kernel. "
                   "Google Colab: Runtime menu, Restart session.)")

import os
from getpass import getpass

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime.accounts import AccountNotFoundError
from qiskit_ibm_runtime.fake_provider import FakeFez

class SimulatedBackend(FakeFez):
    """A simulated copy of the IBM processor ibm_fez. It runs where this notebook runs, and nothing is sent."""

    def __repr__(self):
        return f"<{type(self).__name__}('{self.name}')>"


SIMULATED_BACKEND = SimulatedBackend()


class SimulatedService:
    """Stands in for QiskitRuntimeService when nothing is sent to IBM.

    It answers the same calls and always hands out the simulated processor.
    """

    def backends(self, **filters):
        return [SIMULATED_BACKEND]

    def backend(self, name=None, **filters):
        return SIMULATED_BACKEND

    def least_busy(self, **filters):
        return SIMULATED_BACKEND


def connect(account):
    """Return the entry point for one account. Every hardware cell calls this.

    connect("ibm") returns a service. service.least_busy(...) then gives a backend.
        USE_HARDWARE = True and HARDWARE = "ibm": your own IBM Quantum account.
        Any other setting: a simulated IBM processor. No account is used.
    connect("qbraid") returns a qBraid provider, or None when that route is off.
        USE_HARDWARE = True and HARDWARE = "qbraid": your qBraid account.
        Any other setting: None. Nothing is sent.
    """
    route = str(HARDWARE).strip().lower()
    if not isinstance(USE_HARDWARE, bool) or route not in ("ibm", "qbraid"):
        raise ValueError("Check the first code cell: USE_HARDWARE must be True or False, "
                         'and HARDWARE must be "ibm" or "qbraid".')
    if account not in ("ibm", "qbraid"):
        raise ValueError('Call connect("ibm") or connect("qbraid").')
    live = USE_HARDWARE and route == account
    why = "USE_HARDWARE is False" if not USE_HARDWARE else f'HARDWARE is "{route}"'

    if account == "ibm":
        if not live:
            print(f"Simulator: this cell runs on {SIMULATED_BACKEND.name}, a simulated IBM "
                  f"processor ({why}). No account is used and nothing is sent.")
            return SimulatedService()
        try:
            service = QiskitRuntimeService()
        except AccountNotFoundError:
            print("No IBM Quantum account is saved on this computer. Paste your API key and "
                  "your instance CRN in the boxes. They stay hidden. They are saved in your "
                  "home folder, never in this notebook.")
            QiskitRuntimeService.save_account(
                token=getpass("IBM Quantum API key: "),
                instance=getpass("Instance CRN (press Enter to skip): ") or None,
                set_as_default=True, overwrite=True)
            service = QiskitRuntimeService()
        print("Real hardware: this cell uses your own IBM Quantum account and your quantum time.")
        return service

    if not live:
        print(f"Skipped: this cell uses a qBraid account, the second option ({why}). "
              "Nothing is sent.")
        return None
    from qbraid.runtime import QbraidProvider
    if not (os.environ.get("QBRAID_API_KEY")
            or os.path.exists(os.path.expanduser("~/.qbraid/qbraidrc"))):
        print("No qBraid API key is saved on this computer. Paste it in the box. It stays "
              "hidden and is kept only until this kernel stops.")
        os.environ["QBRAID_API_KEY"] = getpass("qBraid API key: ")
    if QBRAID_DEVICE == "PASTE_DEVICE_ID":
        print('QBRAID_DEVICE is still "PASTE_DEVICE_ID". Copy the id of an online device from '
              "the list this cell prints into the first code cell, run that cell, then run "
              "this cell again.")
    print("Real hardware, second option: this cell uses your qBraid account. Device:", QBRAID_DEVICE)
    return QbraidProvider()

The setup check prints what is installed and where circuits will run. If the versions match the tested ones, you are ready.

In [ ]:
# Setup check: what is installed, and where circuits will run.
import qiskit, qiskit_aer, qiskit_ibm_runtime   # an error here means an install went wrong

print("Python", sys.version.split()[0])
for _name, _pin in PINNED.items():
    print(f"{_name} {_found(_name)}  (tested with {_pin})")
print("USE_HARDWARE =", USE_HARDWARE, "| HARDWARE =", repr(HARDWARE),
      "| QBRAID_DEVICE =", repr(QBRAID_DEVICE))
print("Simulated IBM processor:", SIMULATED_BACKEND.name, "with",
      SIMULATED_BACKEND.num_qubits, "qubits")

## Join the Discord and register

*Slide 2.*

- **Discord.** Join the server at https://discord.gg/FBNzRcxMr. Questions between sessions go there.
- **Registration.** Not registered yet? Fill in the form at https://forms.gle/NC9o1iRk3WggLuTH7.

On the slide the two links are QR codes, small square patterns that a phone camera reads as a link. Discord is the chat service where the club answers questions.

## New tonight?

*Slide 4.*

- **IBM Quantum Platform.** quantum.cloud.ibm.com. Open Plan instance and an API key. This is where your circuits run.
- **qBraid, the second option.** account.qbraid.com. Sign up with the email you registered with. Access is sent there.
- **Step by step.** The Accounts section of the Session 1 notebook, on the event page.

No account yet? The lab also runs on the simulator. Open tonight's notebook and run the first cell.

You need an account only to run on a real quantum computer. Everything in this notebook runs without one, on the simulator, so you can do the whole lab tonight and set up an account later.

Some words from the first card. IBM Quantum Platform is the website where you manage your IBM Quantum account. An API key is a secret string of characters that lets a program, such as this notebook, act for your account. A plan is the kind of access an account has, and the Open Plan is the one you do not pay for. An instance is your share of access under a plan. Each instance has a CRN, short for cloud resource name: the long identifier that IBM Quantum Platform lists with your instances. With `USE_HARDWARE = True`, the first hardware cell asks for it, together with the API key, when no account is saved yet.

IBM may ask for a credit card when you create the account. The card is not charged unless you create a Pay-As-You-Go instance and use it to send work to a quantum computer. If you cannot or do not want to give one, ask an organizer on Discord. Challenges 3 and 4 need no account.

In the Accounts section of the Session 1 notebook the steps are cells you can run: IBM Quantum Platform first, then the second option. A job is one request to run circuits. Jobs on IBM processors always go through your own IBM Quantum account. The second option does not replace that account. It adds the other quantum computers that qBraid offers.

In this notebook "the first cell" means the three code cells of the Setup section above. If they ran without an error, you are ready. The setup check prints `fake_fez`, the name of the simulated processor, and the name appears in the output of several cells.

## Part 1: The seminar ideas

This part follows tonight's slides in order. Each heading is the title of a slide. Under it you find a short recap, the math from the slide, and often a small cell to run. The sections called "Derivation" show the steps behind a result, one at a time.

**If you missed the earlier sessions.** Tonight builds on a few ideas from earlier in the week. Here they are, each in a sentence or two.

- **Qubit.** The quantum version of a bit. When you measure a qubit, you read a 0 or a 1.
- **State, ket and amplitude.** The state of a qubit is its full description, everything you need in order to predict what a measurement will give. $\lvert 0 \rangle$ and $\lvert 1 \rangle$ are the two states in which a measurement is certain to give 0 or 1. A symbol of this shape is called a ket. A general state is a blend $\alpha\lvert 0 \rangle + \beta\lvert 1 \rangle$. The numbers $\alpha$ and $\beta$ are the amplitudes. You read 0 with probability $\lvert \alpha \rvert^2$ and 1 with probability $\lvert \beta \rvert^2$, and the two probabilities add up to 1. An amplitude can be negative, and in general it is a complex number, a number of the form $x + iy$ with $i^2 = -1$. $\lvert \alpha \rvert$ is its size, $\sqrt{x^2 + y^2}$, which for an ordinary number is the number without its sign.
- **Superposition and phase.** A state in which both amplitudes are different from zero is a superposition. Two examples are $\lvert + \rangle = \frac{\lvert 0 \rangle + \lvert 1 \rangle}{\sqrt{2}}$ and $\lvert - \rangle = \frac{\lvert 0 \rangle - \lvert 1 \rangle}{\sqrt{2}}$. Both give 0 or 1 with probability one half. They differ only in the sign of one amplitude. Such a sign is called a phase. A measurement made right away cannot see it, but later gates can.
- **Gate.** An operation that changes the state without measuring it. X flips a qubit between $\lvert 0 \rangle$ and $\lvert 1 \rangle$. That is called a bit flip. Z puts a minus sign on the amplitude of $\lvert 1 \rangle$. That is called a phase flip. H, the Hadamard gate, turns $\lvert 0 \rangle$ into $\lvert + \rangle$ and $\lvert 1 \rangle$ into $\lvert - \rangle$, and a second H undoes the first. CX acts on two qubits: it flips the second one, the target, when the first one, the control, is 1. CZ puts a minus sign on the amplitude of the result in which both qubits are 1. SWAP exchanges the states of two qubits.
- **Several qubits and entanglement.** The state of $n$ qubits has one amplitude for each of the $2^n$ possible results. One H and one CX turn two qubits in 0 into the Bell pair $\frac{\lvert 00 \rangle + \lvert 11 \rangle}{\sqrt{2}}$. Each of its qubits gives 0 or 1 at random, but the two always give the same bit. A state like this, in which no qubit has a state of its own, is called entangled. A GHZ state is the same thing on more qubits: all zeros or all ones.
- **Circuit, shot and counts.** A circuit is a list of gates applied to qubits, usually ending with a measurement. One run of a circuit is called a shot. It gives one bitstring, a string of 0s and 1s with one character per qubit. The counts say how many shots gave each bitstring. Qiskit numbers the qubits from 0 and prints qubit 0 as the right-most character.
- **Expectation value.** The average of a measured quantity over many shots. Tonight's quantity for two qubits is called $ZZ$. It is $+1$ for a shot in which the two qubits gave the same bit and $-1$ for a shot in which they gave different bits. For a perfect Bell pair every shot counts $+1$, so the expectation value is 1. Errors pull it down toward 0.
- **Backend, simulator and primitives.** A backend is the machine, real or simulated, that runs a circuit. A simulator is a program that imitates a quantum computer on an ordinary computer. A primitive is Qiskit's standard interface for running circuits. There are two. The Sampler returns the measured bitstrings. The Estimator returns expectation values.

The cell below makes a Bell pair and uses both primitives on an ideal simulator, one without any noise.

In [ ]:
from qiskit import QuantumCircuit
from qiskit.primitives import StatevectorEstimator, StatevectorSampler
from qiskit.quantum_info import SparsePauliOp

pair = QuantumCircuit(2)             # a circuit with two qubits, both starting in 0
pair.h(0)                            # H on qubit 0: an equal superposition of 0 and 1
pair.cx(0, 1)                        # CX with qubit 0 as the control: the two qubits are now a Bell pair

# The Sampler runs the circuit for a number of shots and returns what was measured.
measured_pair = pair.measure_all(inplace=False)                  # a copy of the circuit, with measurements
ideal_sampler = StatevectorSampler(seed=5)                       # the seed fixes the random numbers
pair_counts = ideal_sampler.run([measured_pair], shots=1000).result()[0].data.meas.get_counts()
print("counts:", pair_counts)

# The expectation value of ZZ from the counts: +1 for a shot with two equal bits, -1 otherwise.
equal = pair_counts.get("00", 0) + pair_counts.get("11", 0)
print("average of ZZ over the 1,000 shots:", (equal - (1000 - equal)) / 1000)

# The Estimator returns the expectation value directly. It takes the circuit without measurements.
zz = SparsePauliOp("ZZ")             # the quantity ZZ, in the form the Estimator expects
value = StatevectorEstimator().run([(pair, zz)]).result()[0].data.evs
print("expectation value of ZZ from the Estimator:", round(float(value), 6))

The 1,000 shots split into 521 of `00` and 479 of `11`, and no shot gave `01` or `10`. So the average of $ZZ$ is exactly 1, and the Estimator returns the same value. A quantity handed to the Estimator, such as `zz`, is called an observable.

A real processor would also give a few `01` and `10`, and the average would fall below 1. How far below, and what can be done about it, is the subject of tonight.

### Inside an IBM quantum processor

*Slides 6 to 10.*

- **The qubit.** A superconducting circuit on a chip. Its two lowest energy levels are 0 and 1, and the cold keeps it in place.
- **The gates.** Shaped pulses: a burst of microwaves for one qubit, a pulse on the link for two. Two-qubit gates are slower and noisier.
- **The wiring.** Each qubit couples to at most three neighbors on tonight's heavy-hex processor.
- **The readout.** A resonator beside each qubit responds differently for 0 and for 1.

Slides 7 to 10 open the four cards one at a time, each with a picture. Here are the same four in words.

An IBM qubit is a small electrical circuit made of a superconductor, a metal that carries current without resistance when it is cold enough. Such a circuit can hold only certain amounts of energy, called its levels. The two lowest levels serve as $\lvert 0 \rangle$ and $\lvert 1 \rangle$. The next step up is a little smaller than the first, so a pulse tuned to the first step leaves the higher levels alone: two levels are enough. This kind of qubit is called a transmon.

A one-qubit gate is a microwave pulse: a short burst of radio waves, sent to the qubit through a wire. The strength and the length of the pulse decide how far the state turns. A Z gate needs no pulse at all: the electronics shift the timing of the later pulses instead. A two-qubit gate is a pulse on the coupler, the link between two neighboring qubits.

A two-qubit gate works only between two qubits that are coupled, which means physically connected on the chip. On the processor used tonight each qubit is coupled to at most three others, as the cell below counts. The pattern is called heavy-hex: a honeycomb of six-sided rings with one extra qubit on every side. IBM's Heron processors, such as ibm_fez, are built this way. IBM's newer Nighthawk processors use a grid instead, which gives each qubit more neighbors.

Beside each qubit sits a resonator, a second small circuit that is used to measure it. A microwave signal sent to the resonator comes back slightly different depending on whether the qubit is in 0 or in 1.

After the math of slides 11 and 12, the slides come back to these parts in turn: the qubit's temperature and lifetimes, the readout, the gates with their error budget, and what the wiring does to your circuit.

The cell below looks at the wiring of the simulated processor of this notebook, a copy of the IBM processor ibm_fez. It counts the neighbors of every qubit and draws the qubit map. Each dot is a qubit. Each line joins two qubits that are coupled.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

chip = SIMULATED_BACKEND                        # the simulated copy of the IBM processor ibm_fez
pairs = chip.coupling_map.get_edges()           # every coupled pair is listed twice, once in each direction

neighbors = {q: set() for q in range(chip.num_qubits)}
for first, second in pairs:
    neighbors[first].add(second)
    neighbors[second].add(first)
print("qubits:", chip.num_qubits, "| coupled pairs:", len(pairs) // 2)
for size in (1, 2, 3):
    print("number of neighbors:", size, "| qubits:", sum(1 for q in neighbors if len(neighbors[q]) == size))


def draw_qubit_map(processor, chosen=(), title="Qubit map of the simulated processor"):
    """Draw every qubit as a numbered dot and every coupled pair as a line. Qubits in `chosen` are orange."""
    # Where to draw each qubit. This processor has 8 rows of 16 qubits, joined by short rows of 4 qubits.
    position = {}
    for q in range(processor.num_qubits):
        row, k = divmod(q, 20)
        if k < 16:
            position[q] = (k, -2 * row)
        else:
            position[q] = (4 * (k - 16) + (3 if row % 2 == 0 else 1), -2 * row - 1)
    fig, ax = plt.subplots(figsize=(6.5, 6.5))
    for first, second in processor.coupling_map.get_edges():      # one line for each coupled pair
        ax.plot([position[first][0], position[second][0]], [position[first][1], position[second][1]],
                color="0.6", lw=2, zorder=1)
    for q, (x, y) in position.items():                            # one numbered dot for each qubit
        ax.scatter(x, y, s=130, color="tab:orange" if q in chosen else "#1f4e9c", zorder=2)
        ax.text(x, y, str(q), ha="center", va="center", color="white", fontsize=6, zorder=3)
    ax.set_aspect("equal")
    ax.axis("off")
    ax.set_title(title)
    plt.show()


draw_qubit_map(chip)

The processor has 156 qubits and 176 coupled pairs. 100 qubits have two neighbors and 48 have three. The 8 qubits with a single neighbor sit on the edge of the chip. No qubit has more than three. In the drawing the six-sided rings are stretched into rectangles: each closed loop has twelve qubits.

The second card says that two-qubit gates are slower and noisier than one-qubit gates. The next cell reads this from the calibration data of the simulated processor, a record of the real ibm_fez from Feb 26, 2025. For each kind of operation it prints the median error, which is the chance that the operation goes wrong, and the median duration. The median is the middle value: half of the qubits or pairs are better and half are worse. `sx` is a one-qubit gate this processor performs directly. It is half of an X gate: two of them in a row make an X. `cz` is the processor's two-qubit gate. The last line prints the medians of $T_1$ and $T_2$, the two lifetimes that the section "Qubits forget" below explains.

In [ ]:
chip_target = chip.target                       # every operation this processor offers, with its calibration numbers


def median_of(operation, field):
    """The median of one calibration number over all qubits, or pairs, that have this operation."""
    values = [getattr(entry, field) for entry in chip_target[operation].values()]
    return float(np.median(values))


print("operation            median error   median duration")
for label, operation in [("one-qubit gate (sx)", "sx"), ("two-qubit gate (cz)", "cz"), ("readout            ", "measure")]:
    print(label, " ", f"{median_of(operation, 'error'):.5f}", "      ",
          round(median_of(operation, "duration") * 1e9), "nanoseconds")

chip_properties = chip.properties()             # the calibration numbers of the qubits themselves, in seconds
t1_values = [chip_properties.t1(q) for q in range(chip.num_qubits)]
t2_values = [chip_properties.t2(q) for q in range(chip.num_qubits)]
print("median T1:", round(float(np.median(t1_values)) * 1e6), "microseconds | median T2:",
      round(float(np.median(t2_values)) * 1e6), "microseconds")

A nanosecond is a billionth of a second and a microsecond is a millionth. On this processor a two-qubit gate takes 84 nanoseconds against 24 for a one-qubit gate, and it goes wrong about 17 times as often: 0.39% against 0.023%. Readout is the slowest step and the least reliable one: 1,560 nanoseconds and 0.76%. Lab 1 reads the same record qubit by qubit and pair by pair.

### The math for today: tensor products and unitary matrices

*Slide 11.*

- **States multiply.** Two separate qubits combine as $\lvert \psi \rangle \otimes \lvert \phi \rangle$: four amplitudes, and $2^n$ for $n$ qubits. Entangled states, like a Bell pair, do not split this way.
- **Gates are unitary.** Every gate is a matrix with $U^{\dagger} U = I$, so the probabilities always sum to 1.
- **Noise is not.** A real qubit also loses what it stores, and that loss is not unitary. The next steps are about fighting it.

The symbol $\otimes$ is called the tensor product. It joins the descriptions of two separate qubits into one description of the pair. Say the first qubit is in $\lvert \psi \rangle = a\lvert 0 \rangle + b\lvert 1 \rangle$ and the second is in $\lvert \phi \rangle = u\lvert 0 \rangle + v\lvert 1 \rangle$. The tensor product multiplies out like ordinary brackets.

$$\lvert \psi \rangle \otimes \lvert \phi \rangle = au\lvert 00 \rangle + av\lvert 01 \rangle + bu\lvert 10 \rangle + bv\lvert 11 \rangle$$

Two qubits have four amplitudes, three qubits have eight, and $n$ qubits have $2^n$. Not every list of four amplitudes is such a product. The Bell pair $\frac{\lvert 00 \rangle + \lvert 11 \rangle}{\sqrt{2}}$ is not: no choice of $a$, $b$, $u$ and $v$ gives it. That is what entangled means. On this slide and the next, the first qubit is the left character of a label such as $\lvert 01 \rangle$.

A state of $n$ qubits is a column of $2^n$ amplitudes, and a gate is a matrix that multiplies this column. A circuit without noise is one large matrix $U$, the product of the matrices of its gates. The dagger in $U^{\dagger}$ means: exchange rows and columns, and replace every entry by its complex conjugate. The complex conjugate of $x + iy$ is $x - iy$, so for a matrix of ordinary numbers only the exchange matters. $I$ is the identity matrix, the one that changes nothing. A matrix with $U^{\dagger} U = I$ is called unitary. The derivation below shows what follows from that.

Noise is different. When a qubit decays, $\lvert 1 \rangle$ ends up as $\lvert 0 \rangle$, and $\lvert 0 \rangle$ stays $\lvert 0 \rangle$. Two different states have become the same state, so nothing can tell afterwards which one it was. Decay, the process behind $T_1$, and dephasing, the process behind $T_2$, cannot be written as a unitary matrix, and they cannot be undone by running gates backward. The section "Qubits forget" below explains both.

#### Derivation: Tensor products and unitaries, worked

*Slide 12.*

A gate on one of two qubits. $H \otimes I$ means H on the first qubit and nothing on the second. To build its matrix, start from the matrix of H, $\frac{1}{\sqrt{2}}\begin{pmatrix} 1 & 1 \\ 1 & -1 \end{pmatrix}$, and replace each of its four entries by that entry times the 2 by 2 identity matrix $I$. The result is a 4 by 4 matrix written in blocks.

$$H \otimes I = \frac{1}{\sqrt{2}} \begin{pmatrix} I & I \\ I & -I \end{pmatrix}$$

Written out in full, with rows and columns in the order 00, 01, 10, 11:

$$H \otimes I = \frac{1}{\sqrt{2}} \begin{pmatrix} 1 & 0 & 1 & 0 \\ 0 & 1 & 0 & 1 \\ 1 & 0 & -1 & 0 \\ 0 & 1 & 0 & -1 \end{pmatrix}$$

The slide puts the first qubit on the left. Qiskit prints qubit 0 on the right, so its left character is the qubit with the higher number. In a two-qubit Qiskit circuit this matrix is therefore `qc.h(1)`, not `qc.h(0)`.

Unitary means:

$$U^{\dagger} U = I$$

So lengths are kept. $\langle \psi \rvert$, called a bra, is the column of $\lvert \psi \rangle$ written as a row, with every amplitude replaced by its complex conjugate. A bra times a ket is a single number, and $\langle \psi \vert \psi \rangle$ is the sum of the squared sizes of all amplitudes: the total probability, which is 1. After a gate the state is $U\lvert \psi \rangle$, which the slide abbreviates as $\lvert U\psi \rangle$. Its bra is $\langle \psi \rvert U^{\dagger}$.

$$\langle U\psi \vert U\psi \rangle = \langle \psi \rvert U^{\dagger} U \lvert \psi \rangle = \langle \psi \vert \psi \rangle = 1$$

The total probability after a gate equals the total probability before it.

And every gate can be undone. Multiplying by $U^{\dagger}$ after $U$ gives the identity, so $U^{\dagger}$ is the inverse of $U$.

$$U^{-1} = U^{\dagger}$$

To undo a whole circuit, run its gates in reverse order, each one replaced by its dagger. Noise breaks this.

The cell below checks each line with Qiskit.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator

I2 = np.eye(2)                                           # the 2 by 2 identity matrix
slide_matrix = np.block([[I2, I2], [I2, -I2]]) / np.sqrt(2)
print("the slide's H (x) I, in blocks:")
print(np.round(slide_matrix, 3) + 0.0)                   # adding 0.0 prints -0. as 0.

h_on_qubit_1 = QuantumCircuit(2)
h_on_qubit_1.h(1)                                        # qubit 1 is the left character in Qiskit's labels
h_on_qubit_0 = QuantumCircuit(2)
h_on_qubit_0.h(0)                                        # qubit 0 is the right character
print("h(1) gives the slide's matrix:", np.allclose(Operator(h_on_qubit_1).data, slide_matrix))
print("h(0) gives the slide's matrix:", np.allclose(Operator(h_on_qubit_0).data, slide_matrix))

some_gates = QuantumCircuit(2)                           # a circuit without noise is one unitary matrix
some_gates.h(0)
some_gates.cx(0, 1)
some_gates.t(1)                                          # T turns the phase of 1 by an eighth of a full turn
U = Operator(some_gates).data
print("U^dagger U is the identity:", np.allclose(U.conj().T @ U, np.eye(4)))

rng = np.random.default_rng(14)                          # the seed fixes the random numbers
psi = rng.normal(size=4) + 1j * rng.normal(size=4)       # four random complex amplitudes
psi = psi / np.linalg.norm(psi)                          # scaled so that the probabilities sum to 1
print("total probability before and after U:", round(float(np.sum(np.abs(psi) ** 2)), 12),
      "and", round(float(np.sum(np.abs(U @ psi) ** 2)), 12))

undo = some_gates.inverse()                              # the gates in reverse order, each one daggered
print("gates of the circuit:", [instruction.operation.name for instruction in some_gates.data])
print("gates of its inverse:", [instruction.operation.name for instruction in undo.data])
print("the circuit followed by its inverse does nothing:",
      Operator(some_gates.compose(undo)).equiv(Operator(np.eye(4))))

The printed matrix is the block matrix of the slide, where 0.707 is $\frac{1}{\sqrt{2}}$. `h(1)` gives it and `h(0)` does not. The test circuit of three gates is unitary, and a random state has the total probability 1 before and after it. The inverse circuit lists the gates in reverse order. `tdg` is the dagger of the T gate. H and CX are their own daggers, so their names do not change. In Python, `U.conj().T` is $U^{\dagger}$ and `@` multiplies matrices.

### Why the qubits live at $10\,\text{mK}$

*Slide 13.*

- Going from 0 to 1 costs one step of energy. It is Planck's constant $h$ times the qubit's frequency $f$, which is about 5 GHz.
- Heat hands out kicks of energy. A typical kick is Boltzmann's constant $k_B$ times the temperature $T$, in kelvin (K).
- Step and kick are equal at 0.24 K. Room temperature is 1,250 times hotter; the refrigerator, at 10 mK, is 24 times colder, so heat almost never flips the qubit.

The picture on the slide is a temperature scale: room temperature at 300 K, outer space at 2.7 K, the qubit's energy step written as a temperature at 0.24 K, and the refrigerator at 10 mK. As a formula, step and kick are equal at

$$\frac{h f}{k_B} \approx 0.24\,\text{K} \quad \text{for} \; f = 5\,\text{GHz}$$
The two levels of the qubit are separated by a fixed amount of energy, its energy step. The step is $hf$. Here $f$ is the frequency of the qubit, the frequency of the microwave pulse that drives it from 0 to 1. It is about 5 GHz, five billion oscillations per second. $h$ is Planck's constant, a constant of nature.

Heat is the random motion of the surroundings. At a temperature $T$ it comes in typical portions of energy $k_B T$, where $k_B$ is Boltzmann's constant, another constant of nature. $T$ is measured in kelvin (K), a scale that starts at absolute zero, the lowest temperature there is. Room temperature is about 300 K.

Dividing the energy step by $k_B$ turns it into a temperature: 0.24 K. Surroundings at that temperature hand out portions of heat as large as the energy step. Room temperature is about 1,250 times hotter, so at room temperature heat would knock the qubit back and forth between 0 and 1 all the time. That is why the chip sits in a refrigerator at about 10 mK. A millikelvin (mK) is a thousandth of a kelvin, so 10 mK is one hundredth of a kelvin above absolute zero.

One caution. 10 mK is the temperature of the refrigerator. The qubits themselves do not get quite that cold, and on a real processor the chance of finding a qubit excited by heat is far larger than the $4 \times 10^{-11}$ computed below, although it is still small.

#### Derivation: The Boltzmann factor at two temperatures

*Slide 14.*

The Boltzmann factor is $e^{-hf/k_B T}$, where $e \approx 2.718$ is the base of the natural logarithm. It is the chance of finding the qubit in its upper level, divided by the chance of finding it in its lower level, when heat alone decides. A factor near 1 means that heat scrambles the qubit freely. A factor near zero means that the qubit stays where you put it.

Energy step of a $5\,\text{GHz}$ qubit. Multiply Planck's constant, $h = 6.626 \times 10^{-34}$ joule seconds, by the frequency, $5 \times 10^{9}$ oscillations per second. The joule (J) is the unit of energy.

$$hf = 6.626 \times 10^{-34} \times 5 \times 10^{9} = 3.31 \times 10^{-24}\,\text{J}$$

As a temperature. Divide by Boltzmann's constant, $k_B = 1.381 \times 10^{-23}$ joules per kelvin.

$$\frac{hf}{k_B} = \frac{3.31 \times 10^{-24}}{1.381 \times 10^{-23}} = 0.24\,\text{K}$$

At $T = 300\,\text{K}$. The exponent is $0.24 / 300 = 0.0008$.

$$e^{-hf/k_B T} = e^{-0.0008} \approx 0.999$$

At $T = 10\,\text{mK}$. The exponent is $0.24 / 0.010 = 24$.

$$e^{-0.24/0.010} = e^{-24} \approx 4 \times 10^{-11}$$

The cell below repeats the arithmetic and draws the Boltzmann factor against the temperature. Both axes are logarithmic: each mark is a fixed factor away from the next one.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

h = 6.626e-34          # Planck's constant, in joule seconds
k_B = 1.381e-23        # Boltzmann's constant, in joules per kelvin
f = 5e9                # the frequency of the qubit: 5 GHz, in oscillations per second

step_kelvin = h * f / k_B                      # the energy step, written as a temperature
print(f"energy step h f = {h * f:.3g} J, which as a temperature is {step_kelvin:.2f} K")
for temperature, name in [(300, "300 K"), (0.010, "10 mK")]:
    print(f"at {name}: exponent {-step_kelvin / temperature:.3g}, Boltzmann factor {np.exp(-step_kelvin / temperature):.3g}")

kelvin = np.logspace(np.log10(0.005), 3, 300)  # 300 temperatures from 5 mK to 1,000 K
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.loglog(kelvin, np.exp(-step_kelvin / kelvin), color="tab:blue")
for temperature, label, offset in [(300, "300 K: 0.999", (-78, -16)), (0.010, "10 mK: 4e-11", (10, -4))]:
    ax.plot(temperature, np.exp(-step_kelvin / temperature), "o", color="black")
    ax.annotate(label, (temperature, np.exp(-step_kelvin / temperature)), textcoords="offset points", xytext=offset)
ax.set_xlabel("temperature (kelvin)")
ax.set_ylabel("Boltzmann factor")
ax.set_title("Boltzmann factor of a 5 GHz qubit")
ax.grid(True, alpha=0.3)
plt.show()

At 300 K the factor is 0.999: the upper level is almost as likely as the lower one, so the qubit would hold no information. At 10 mK it is `3.81e-11`, which is how Python writes $3.81 \times 10^{-11}$. The slide rounds it to $4 \times 10^{-11}$. Above about 1 K the curve is nearly flat: the factor is already 0.79 at 1 K. Below that it drops steeply: between 1 K and 10 mK the factor falls by ten powers of ten.

### Qubits forget: $T_1$ and $T_2$

*Slide 15.*

$$P_1(t) = e^{-t/T_1}, \quad T_2 \leq 2T_1$$

$T_1$ is how long a 1 stays a 1.
$T_2$ is how long a superposition keeps its phase. It decays the same way, with $T_2$ in place of $T_1$.

Cold is not enough. A qubit in $\lvert 1 \rangle$ holds one energy step more than a qubit in $\lvert 0 \rangle$. Sooner or later it hands that energy to its surroundings and drops to $\lvert 0 \rangle$, even with no heat around. $P_1(t)$ is the probability that a qubit prepared in 1 is still in 1 after a time $t$. It falls exponentially, which means that it shrinks by the same factor in every equal stretch of time. $T_1$ is the time after which $e^{-1} \approx 37\%$ of the 1 is left.

A superposition such as $\lvert + \rangle$ can go wrong in a second way. Its phase can drift, so that the minus sign that separates $\lvert + \rangle$ from $\lvert - \rangle$ becomes uncertain. This is called dephasing, and its time scale is $T_2$. The inequality on the slide says that a phase cannot outlive the energy by much. Decay damages the phase too. As the probability $P_1(t)$ of finding 1 falls like $e^{-t/T_1}$, the amplitude of 1 falls like $e^{-t/(2T_1)}$, because a probability is the squared size of an amplitude, and the phase lives in that amplitude. So even with no other cause of dephasing, $T_2$ can never be longer than $2T_1$.

On the simulated processor the median $T_1$ is 145 microseconds and the median $T_2$ is 88 microseconds, as printed above. A two-qubit gate takes 84 nanoseconds, so about 1,700 of them fit into one $T_1$. Long before that many gates have run, a noticeable part of every 1 has decayed.

The cell below imitates a qubit with $T_1 = 200$ microseconds, the value of the worked example that follows, and $T_2 = 100$ microseconds. It uses Aer, Qiskit's fast simulator, with a noise model. A noise model is a description of what goes wrong, which the simulator applies while it runs a circuit. Here every `id` gate, a gate that does nothing, stands for one microsecond of waiting. The cell runs two tests.

- The $T_1$ test: flip the qubit to 1, wait, measure. It records the fraction of shots that still read 1.
- The $T_2$ test: put the qubit in $\lvert + \rangle$ with an H gate, wait, apply H again, measure. Without noise the second H brings the qubit back to 0 every time. It records the fraction of shots that read 0.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from qiskit import QuantumCircuit
from qiskit_aer.noise import NoiseModel, thermal_relaxation_error
from qiskit_aer.primitives import SamplerV2 as AerSampler    # Aer's own Sampler, under a name of its own

T1 = 200e-6                                   # 200 microseconds, written in seconds
T2 = 100e-6                                   # 100 microseconds
tick = 1e-6                                   # every id gate below stands for a wait of 1 microsecond

decay_noise = NoiseModel()
decay_noise.add_all_qubit_quantum_error(thermal_relaxation_error(T1, T2, tick), "id")
decay_sampler = AerSampler(seed=7, options={"backend_options": {"noise_model": decay_noise}})

decay_shots = 20_000
waits = [0, 20, 50, 100, 150, 200, 300, 400]  # waiting times in microseconds
still_one, back_to_zero = [], []
for wait in waits:
    t1_test = QuantumCircuit(1)
    t1_test.x(0)                              # start in 1
    for _ in range(wait):
        t1_test.id(0)                         # wait one microsecond
    t1_test.measure_all()

    t2_test = QuantumCircuit(1)
    t2_test.h(0)                              # start in the superposition |+>
    for _ in range(wait):
        t2_test.id(0)                         # wait one microsecond
    t2_test.h(0)                              # a second H brings an undisturbed |+> back to 0
    t2_test.measure_all()

    results = decay_sampler.run([t1_test, t2_test], shots=decay_shots).result()
    still_one.append(results[0].data.meas.get_counts().get("1", 0) / decay_shots)
    back_to_zero.append(results[1].data.meas.get_counts().get("0", 0) / decay_shots)

print("wait   still 1   e^(-t/T1)   back to 0   (1 + e^(-t/T2)) / 2")
for wait, p_one, p_zero in zip(waits, still_one, back_to_zero):
    print(f"{wait:4d}   {p_one:.3f}     {np.exp(-wait * 1e-6 / T1):.3f}       {p_zero:.3f}       {(1 + np.exp(-wait * 1e-6 / T2)) / 2:.3f}")

smooth = np.linspace(0, 400, 200)
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(smooth, np.exp(-smooth * 1e-6 / T1), color="tab:blue", label="e^(-t/T1)")
ax.plot(waits, still_one, "o", color="tab:blue", label="T1 test: fraction still 1")
ax.plot(smooth, (1 + np.exp(-smooth * 1e-6 / T2)) / 2, color="tab:orange", label="(1 + e^(-t/T2)) / 2")
ax.plot(waits, back_to_zero, "s", color="tab:orange", label="T2 test: fraction back to 0")
ax.set_ylim(0, 1.05)
ax.set_xlabel("waiting time (microseconds)")
ax.set_ylabel("fraction of 20,000 shots")
ax.set_title("A qubit forgets: T1 = 200 and T2 = 100 microseconds")
ax.legend()
plt.show()

The points lie on the two curves.

- The $T_1$ test follows $e^{-t/T_1}$. After 20 microseconds, 0.909 of the shots still read 1, and after 200 microseconds, one full $T_1$, 0.368 do.
- The $T_2$ test falls from 1 toward one half, not toward 0. A qubit that has lost its phase gives a random bit after the second H, and half of those bits are 0. The curve is $\frac{1}{2}\left( 1 + e^{-t/T_2} \right)$. After 100 microseconds, one full $T_2$, 0.686 of the shots come back to 0.

The small gaps between the points and the curves are shot noise, the scatter you get from counting a limited number of random shots.

#### Derivation: A worked $T_1$ example

*Slide 17.*

Decay law. The probability that a qubit prepared in 1 still reads 1 after a time $t$.

$$P_1(t) = e^{-t/T_1}$$

Illustrative numbers. A lifetime of 200 microseconds, and a circuit that takes 20 microseconds from its first gate to its measurement. The symbol $\mu\text{s}$ stands for microseconds.

$$T_1 = 200\,\mu\text{s}, \quad t = 20\,\mu\text{s}$$

Population left. The population of 1 is the fraction of shots that would still read 1. Only the ratio of the two times matters.

$$e^{-20/200} = e^{-0.1} = 0.905$$

Lost before measurement.

$$1 - 0.905 = 9.5\%$$

The simulated qubit above agrees: its row for 20 microseconds shows 0.909 measured against 0.905 from the formula. So a circuit of this length loses about a tenth of every 1 before it is read, even if every gate is perfect. The time a circuit takes grows with its depth, the number of layers of gates that must run one after another. Depth is time, and time is error. The numbers here are illustrative. Lab 1 reads real values.

### Readout makes mistakes too

*Slide 18.*

|  | Read 0 | Read 1 |
|---|---|---|
| Prepared 0 | 98% | 2% |
| Prepared 1 | 4% | 96% |

- The off-diagonal entries are the readout error.
- They can be measured, then partly undone.
- The numbers here are illustrative.

Measuring a qubit is a physical process too, and it can report the wrong bit. Read the table by rows. A qubit prepared in 0 is read as 0 in 98% of the shots and as 1 in 2%. A qubit prepared in 1 is read as 0 in 4% of the shots and as 1 in 96%. The entries off the diagonal, 2% and 4%, are the wrong readings.

This error is simpler to handle than the others tonight, because it can be treated as an ordinary mistake in the bit that is reported. That is why it can be measured directly: prepare 0 many times and count what is read, then do the same with 1. Once the table is known, its effect on a result can be undone. The undoing is partial because it corrects the fractions of 0s and 1s, never a single shot: nobody can say which of the shots were misread.

#### Derivation: Undoing readout error with a matrix

*Slide 19.*

Call $p_0$ and $p_1$ the true probabilities of 0 and 1, and $p_0'$ and $p_1'$ the fractions that are read.

Measured equals $M$ times true. A 0 is read in two cases: a true 0 that is read correctly, or a true 1 that is misread. So $p_0' = 0.98\,p_0 + 0.04\,p_1$. In the same way $p_1' = 0.02\,p_0 + 0.96\,p_1$. A matrix is a table of numbers. It multiplies a column of numbers one row at a time: the entries of a row are multiplied with the entries of the column in pairs and added, which gives one entry of the result. Written with a matrix $M$, the two equations become one:

$$\begin{pmatrix} p_0' \\ p_1' \end{pmatrix} = \begin{pmatrix} 0.98 & 0.04 \\ 0.02 & 0.96 \end{pmatrix}\begin{pmatrix} p_0 \\ p_1 \end{pmatrix}$$

Each column of $M$ is one row of the table. The first column says how a prepared 0 is read and the second column says how a prepared 1 is read.

Invert the matrix. $M^{-1}$ is the matrix that undoes $M$. The number $\det M$ is called the determinant of $M$: the product of the two diagonal entries minus the product of the other two, $0.98 \times 0.96 - 0.04 \times 0.02 = 0.94$.

$$\begin{pmatrix} p_0 \\ p_1 \end{pmatrix} = M^{-1}\begin{pmatrix} p_0' \\ p_1' \end{pmatrix}, \quad \det M = 0.94$$

For a 2 by 2 matrix the inverse has a short recipe: exchange the two diagonal entries, change the sign of the other two, and divide everything by the determinant.

$$M^{-1} = \frac{1}{0.94}\begin{pmatrix} 0.96 & -0.04 \\ -0.02 & 0.98 \end{pmatrix}$$

Example: measured 0.51 and 0.49. The top row of $M^{-1}$ gives the true probability of 0.

$$p_0 = \frac{0.96 \times 0.51 - 0.04 \times 0.49}{0.94} = 0.50$$

And the other outcome. The bottom row gives the true probability of 1.

$$p_1 = \frac{0.98 \times 0.49 - 0.02 \times 0.51}{0.94} = 0.50$$

The measured split of 51 to 49 was a perfect 50 to 50 before the readout error.

The cell below goes through the whole procedure on a simulator whose readout follows the table. It first calibrates, which means that it measures the table. Then it measures the state $\lvert + \rangle$, whose true probabilities are one half each. Then it solves the matrix equation for the true probabilities.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit_aer.noise import NoiseModel, ReadoutError
from qiskit_aer.primitives import SamplerV2 as AerSampler

table = [[0.98, 0.02],                        # prepared 0: read 0, read 1
         [0.04, 0.96]]                        # prepared 1: read 0, read 1
readout_noise = NoiseModel()
readout_noise.add_all_qubit_readout_error(ReadoutError(table))
readout_sampler = AerSampler(seed=7, options={"backend_options": {"noise_model": readout_noise}})
readout_shots = 100_000


def read_fractions(circuit):
    """Measure a one-qubit circuit and return the fractions of shots that read 0 and 1."""
    measured_circuit = circuit.measure_all(inplace=False)
    read = readout_sampler.run([measured_circuit], shots=readout_shots).result()[0].data.meas.get_counts()
    return np.array([read.get("0", 0), read.get("1", 0)]) / readout_shots


prepare_0 = QuantumCircuit(1)                 # a qubit left in 0
prepare_1 = QuantumCircuit(1)
prepare_1.x(0)                                # a qubit flipped to 1
prepare_plus = QuantumCircuit(1)
prepare_plus.h(0)                             # the superposition |+>: 0 and 1 with probability one half each

# Step 1, calibrate: prepare 0, then 1, and record how each of them is read.
calibration = np.array([read_fractions(prepare_0), read_fractions(prepare_1)])
print("measured table (rows: prepared 0, prepared 1):")
print(calibration)

# Step 2: measure the state you care about.
measured = read_fractions(prepare_plus)
print("measured for |+>:", measured)

# Step 3, undo the readout error: solve  measured = M @ true  for the true probabilities.
M = calibration.T                             # the columns of M are the rows of the table
corrected = np.linalg.solve(M, measured)
print("corrected:       ", np.round(corrected, 4))
print("determinant of M:", round(float(np.linalg.det(M)), 4))

The measured table is close to the one on the slide: 0.980 and 0.020 in the first row, 0.039 and 0.961 in the second. The state $\lvert + \rangle$ is read as 0.513 and 0.487, the slide's 0.51 and 0.49. The correction returns 0.503 and 0.497, which is 0.50 and 0.50 to two decimals. What remains is shot noise in the three measurements.

Two notes on the code. `calibration.T` swaps rows and columns, because the table has one row per prepared state and $M$ has one column per prepared state. `np.linalg.solve(M, measured)` finds the true probabilities without computing $M^{-1}$ first, which is the more accurate way to solve such an equation.

### Two-qubit gates set your error budget

*Slide 20.*

$$F \approx \left( 1 - \varepsilon \right)^{N}$$

$F$, the fidelity, is the chance that no two-qubit gate fails. With an illustrative $\varepsilon = 0.5\%$ per gate: $N = 50$ keeps 78%, $N = 100$ keeps 61%, $N = 200$ keeps 37%.

$\varepsilon$ is the probability that one two-qubit gate goes wrong, and $N$ is the number of two-qubit gates in the transpiled circuit, the version of your circuit that the processor runs (the next section explains it). If the failures are independent of each other, the chance that all $N$ gates work is the product of $N$ factors $1 - \varepsilon$. $F$ stands for fidelity, a number between 0 and 1 that says how close a result is to the ideal one.

The arithmetic: $0.995^{50} = 0.778$, $0.995^{100} = 0.606$ and $0.995^{200} = 0.367$. The value of $\varepsilon$ is illustrative. On the simulated processor the median two-qubit error is 0.39%, as printed earlier.

This is a rough estimate. It leaves out one-qubit gates, readout errors and decay. It counts two-qubit gates because they are typically the most error-prone gates on a processor, so their number is the one to keep small. Lab 2 does that, and so does tonight's challenge.

The cell below draws the estimate against the number of two-qubit gates, with the three numbers of the slide marked.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

eps = 0.005                                    # illustrative chance that one two-qubit gate goes wrong
gate_counts = np.arange(0, 301)                # N from 0 to 300 two-qubit gates
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(gate_counts, (1 - eps) ** gate_counts, color="tab:blue")
for marked in (50, 100, 200):
    kept = (1 - eps) ** marked
    print("N =", marked, "keeps", round(kept, 3))
    ax.plot(marked, kept, "o", color="black")
    ax.annotate(f"N = {marked}: {kept:.2f}", (marked, kept), textcoords="offset points", xytext=(8, 4))
print("first N that keeps less than one half:", int(np.argmax((1 - eps) ** gate_counts < 0.5)))
ax.set_ylim(0, 1.05)
ax.set_xlabel("number of two-qubit gates N")
ax.set_ylabel("estimated fidelity")
ax.set_title("Error budget for eps = 0.005 per two-qubit gate")
ax.grid(True, alpha=0.3)
plt.show()

Every added gate multiplies the estimate by the same factor, 0.995. After 139 gates less than half is left. The only ways to do better are fewer gates or better gates, and the transpiler controls the first.

### The transpiler rewrites your circuit for one machine

*Slide 22.*

- Gates are rewritten into the processor's own gate set.
- Qubits are assigned to physical positions.
- Interactions between non-neighbors are routed with SWAPs.

You write a circuit with any gates you like, between any qubits you like. A processor can do much less. It performs only a handful of gates directly, called its native gates or its gate set, and its two-qubit gate works only between coupled neighbors. The transpiler is the program that closes this gap. Transpiling a circuit means rewriting it for one particular backend. The three bullets are its three jobs.

- **Rewriting.** Each of your gates is replaced by a short sequence of native gates with the same effect. On the simulated processor the native gates are `rz`, `sx` and `x` on one qubit and `cz` on two.
- **Layout.** Each qubit of your circuit is placed on one physical qubit of the chip. This assignment is called the layout.
- **Routing.** When two of your qubits must interact and their physical qubits are not neighbors, SWAP gates move the states across the chip until they are.

The picture on the slide is the qubit map of a 27-qubit heavy-hex processor, the layout of IBM's older Falcon processors. The map drawn above, under "Inside an IBM quantum processor", shows the same pattern with 156 qubits.

In Qiskit the transpiler is run through a pass manager, an object that runs the steps of the transpiler, called passes, one after another. `generate_preset_pass_manager(optimization_level=..., backend=...)` builds one for a backend, and its `run` method returns the rewritten circuit. The optimization level, from 0 to 3, says how hard the transpiler works to shorten the result. Some of its steps try things at random. `seed_transpiler` fixes those random choices, so that the same call gives the same circuit every time.

The cell below transpiles a small circuit for the simulated processor. The circuit has five qubits, and qubit 0 interacts with each of the other four. It prints the gates before and after, and the physical qubits the five qubits sit on at the start and at the end. Then it draws the transpiled circuit. `idle_wires=False` hides the physical qubits that are not used.

In [ ]:
from qiskit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager

star = QuantumCircuit(5)                       # qubit 0 has to interact with four other qubits
star.h(0)
for other in range(1, 5):
    star.cx(0, other)

star_pm = generate_preset_pass_manager(optimization_level=1, backend=SIMULATED_BACKEND, seed_transpiler=0)
star_isa = star_pm.run(star)                   # the transpiled circuit

print("gates you wrote:        ", dict(star.count_ops()))
print("gates after transpiling:", dict(star_isa.count_ops()))
print("physical qubits at the start:", star_isa.layout.initial_index_layout(filter_ancillas=True))
print("physical qubits at the end:  ", star_isa.layout.final_index_layout())
star_isa.draw("mpl", idle_wires=False, fold=-1, scale=0.6)

All three jobs are visible.

- **Rewriting.** `h` and `cx` are gone. The transpiled circuit contains only `rz`, `sx` and `cz`. In the drawing `rz` and `sx` are the boxes, with `sx` drawn as the square root of X, and a `cz` is two dots joined by a vertical line. "Global Phase" at the top is a factor on the whole state. It changes no measurement.
- **Layout.** Your qubits 0 to 4 were placed on the physical qubits 23, 16, 24, 22 and 37. The labels at the left of the drawing say the same: `q_0` sits on 23, and so on.
- **Routing.** Four CX gates became ten CZ gates. Find qubit 23 on the map above. It has three neighbors: 16, 22 and 24, and three of the four partners of your qubit 0 were placed there. The fourth partner sits on qubit 37, three steps away: from 23 to 24, to 25, to 37. Two SWAPs bring the two together. In the drawing they are the two groups of three CZ gates, one between 25 and 37 and one between 23 and 24. Each SWAP costs three two-qubit gates, so the circuit has $4 + 6 = 10$.

Physical qubit 25 is not one of your five. The transpiler borrowed it as a stepping stone, and the drawing calls it `ancilla`. The SWAPs moved states around, which is why the second list of positions differs from the first.

A transpiled circuit is also called an ISA circuit, for "instruction set architecture": a circuit written in the instructions one backend understands. The name `isa` in tonight's lab code comes from there.

This result is not the best one possible. Change `optimization_level=1` to `optimization_level=2` and run the cell again: the transpiler then finds a layout that needs only one SWAP, and the count of two-qubit gates falls to 7. Searching for a good version is Lab 2.

#### Derivation: What routing costs

*Slide 24.*

One SWAP. A SWAP is not a native gate. It is built from three CX gates, with the middle one pointing the other way.

$$\text{SWAP} = 3\,\text{CX}$$

To see why, write the two bits as $x$ and $y$. A CX replaces its target bit by the XOR of the two bits, written $x \oplus y$, which is 1 when exactly one of the two bits is 1. The first CX has the first bit as its control, the second CX has the second bit as its control, and the third is like the first.

$$(x, \; y) \rightarrow (x, \; x \oplus y) \rightarrow (y, \; x \oplus y) \rightarrow (y, \; x)$$

The middle step uses $x \oplus (x \oplus y) = y$ and the last step uses $(x \oplus y) \oplus y = x$, because any bit XORed with itself gives 0. On a processor whose native gate is CZ, each CX becomes one CZ and a few one-qubit gates, so the number of two-qubit gates stays the same.

Two qubits $d$ steps apart. Neighbors are $d = 1$ step apart and need no SWAP. Each SWAP moves one of the two qubits one step closer, so $d - 1$ SWAPs make them neighbors.

$$(d - 1)\,\text{SWAPs} = 3(d - 1)\,\text{CX}$$

Example, $d = 4$. Three SWAPs are needed.

$$3 \times 3 = 9\ \text{extra CX}$$

Fidelity cost at $\varepsilon = 0.5\%$. Fidelity is a number between 0 and 1 that says how close a result is to the ideal one. Suppose each two-qubit gate goes wrong with probability $\varepsilon = 0.005$. The chance that nine of them in a row all work is the product of nine factors $1 - \varepsilon$.

$$0.995^{9} = 0.956$$

One interaction between far-apart qubits can cost several percent before the circuit has done anything useful. In the star circuit above the far partner was $d = 3$ steps away, which cost $3 \times 2 = 6$ extra gates.

The cell below checks the first line and then counts. It uses a made-up processor with six qubits in a row, each coupled only to the next one, and CX as its native two-qubit gate. The circuit is a single CX between qubit 0 and a qubit $d$ steps away. In the first count the layout is pinned: qubit $k$ of the circuit must sit on physical qubit $k$. In the second count the transpiler may choose the layout.

In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator
from qiskit.transpiler import CouplingMap, generate_preset_pass_manager

three_cx = QuantumCircuit(2)
three_cx.cx(0, 1)
three_cx.cx(1, 0)                              # the middle CX points the other way
three_cx.cx(0, 1)
one_swap = QuantumCircuit(2)
one_swap.swap(0, 1)
print("three CX gates do what one SWAP does:", Operator(three_cx).equiv(Operator(one_swap)))

row_of_six = CouplingMap.from_line(6)          # six qubits in a row
pinned_pm = generate_preset_pass_manager(
    optimization_level=1, coupling_map=row_of_six, basis_gates=["cx", "rz", "sx", "x"],
    initial_layout=[0, 1, 2, 3, 4, 5],         # qubit k of the circuit must sit on physical qubit k
    seed_transpiler=0)
free_pm = generate_preset_pass_manager(
    optimization_level=1, coupling_map=row_of_six, basis_gates=["cx", "rz", "sx", "x"],
    seed_transpiler=0)                         # the transpiler may choose the layout

print("steps apart d | CX gates, layout pinned | extra CX | 3 (d - 1) | CX gates, layout free")
for steps in range(1, 6):
    far = QuantumCircuit(6)
    far.cx(0, steps)                           # one CX between two qubits that sit `steps` apart
    pinned = pinned_pm.run(far).count_ops()["cx"]
    free = free_pm.run(far).count_ops()["cx"]
    print(f"      {steps}       |           {pinned:2d}            |    {pinned - 1:2d}    |    {3 * (steps - 1):2d}     |          {free}")

`Operator` turns a circuit into its matrix, and the first line says that the two matrices agree. In the table, the pinned circuit needs one CX for the interaction itself plus 0, 3, 6, 9 and 12 extra ones, exactly $3(d - 1)$. With the layout free, the transpiler places the two qubits side by side and needs a single CX every time.

So the layout decides the cost. This is why two people can transpile the same circuit and get different gate counts. In a larger circuit every qubit has several partners, not every pair can sit side by side, and the transpiler has to search for a layout with few SWAPs. That search is Lab 2, and it is the heart of tonight's challenge.

### Suppress, mitigate, correct: three different jobs

*Slide 25.*

|  | When it acts | Examples | What it costs |
|---|---|---|---|
| Suppression | Before and during the run | Dynamical decoupling, Pauli twirling, a better layout | Almost nothing |
| Mitigation | After the run, on the results | Readout correction, zero-noise extrapolation | More shots and more circuits |
| Correction | During the run, on the qubits | The bit-flip code, the surface code | Many more qubits |

These are three different ways to deal with errors.

- **Suppression** lowers the number of errors that happen. It changes how the circuit is run.
- **Mitigation** stops no error. It estimates what the errors did to an average, such as an expectation value, and removes that effect afterwards with arithmetic.
- **Correction** detects errors and repairs them while the circuit is still running.

The first two work on today's processors. The third is the second half of tonight.

The examples in the table, in a sentence each. Dynamical decoupling: while a qubit waits for other qubits to finish, it is given extra gates that add up to nothing, and these keep slow drifts of its phase from building up. Pauli twirling: the circuit is run in several versions, each with extra one-qubit gates, chosen at random so that the ideal result stays the same. The extra gates are Pauli gates, named after the physicist Wolfgang Pauli: X, Y, Z, or no gate at all. Y acts like a Z followed by an X, up to a factor that changes no measurement. Over the versions, errors that would push the result in one direction turn into errors that partly average out. A better layout means fewer SWAPs on better qubits, which is Lab 2. Readout correction is the matrix inversion you ran above. Zero-noise extrapolation and the two codes follow on the next slides.

One remark on the last column. "Almost nothing" is right for a better layout and for dynamical decoupling. Pauli twirling runs several versions of the circuit, so it does cost extra circuits.

### Mitigation: better answers from a noisy machine

*Slide 26.*

- Readout correction undoes measurement errors.
- Zero-noise extrapolation runs the circuit at amplified noise and extrapolates back.
- Suppression runs alongside it: dynamical decoupling protects idle qubits.

Mitigation does not stop errors from happening. The processor is as noisy as before. What changes is how many circuits are run and what is done with their results: mitigation spends more shots and more circuits to reduce the bias in the final number. Bias is the part of an error that does not shrink when you take more shots. If noise pulls an expectation value from 1 down to 0.93, a million shots tell you 0.93 very precisely, and it is still the wrong number.

Zero-noise extrapolation sounds odd at first. You cannot run a circuit with less noise than the processor has, but you can run it with more. So you measure the same quantity at the normal noise level and at a deliberately raised one, and follow the trend back to the point where the noise would be zero. To extrapolate means to continue a trend beyond the points you measured. The derivation below is the simplest version.

Qiskit Runtime is the service that runs circuits on IBM processors, and qiskit-ibm-runtime is the Python package for it. There, these methods are options on the primitives: you set an option on the Estimator, and it runs the extra circuits and does the arithmetic. Lab 3 switches them on with three lines.

#### Derivation: Zero-noise extrapolation in four lines

*Slide 27.*

Call $\lambda$ the noise scale. $\lambda = 1$ is the processor as it is, $\lambda = 3$ is three times as much noise, and $\lambda = 0$ would be no noise at all. $E(\lambda)$ is the expectation value measured at the scale $\lambda$.

Measure at noise scales 1 and 3.

$$E(1), \quad E(3)$$

Assume a straight line in the noise. $E_0$ is the value at zero noise, the one you want. $a$ is the slope: how much one unit of noise changes the result.

$$E(\lambda) = E_0 + a\lambda$$

Extrapolate to zero noise. The two measurements give two equations, $E(1) = E_0 + a$ and $E(3) = E_0 + 3a$. Three times the first minus the second is $3E(1) - E(3) = 2E_0$, which no longer contains $a$.

$$E_0 = \frac{3E(1) - E(3)}{2}$$

Example. Tripling the noise lowered the result from 0.80 to 0.52.

$$E(1) = 0.80, \; E(3) = 0.52: \quad E_0 = \frac{2.40 - 0.52}{2} = 0.94$$

The straight line is the simplest choice. Real implementations measure at several noise scales and fit other curves. The price is more circuits and a result that scatters more, because the shot noise of every measurement enters the formula.

How do you triple the noise? One way is called gate folding: write every CX three times. Two CX gates in a row cancel, so three do the same job as one, but the qubits pass through the noisy gate three times.

The cell below tries this on a Bell pair. Its simulator has one kind of noise: every CX goes wrong with a probability of 5%, and when it does, the two qubits are left in a completely random state. The cell measures the expectation value of $ZZ$ at the scales 1 and 3 and extrapolates. It uses Aer's own Estimator, which computes averages exactly, without shot noise, so the numbers are the same on every run.

In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer.noise import NoiseModel, depolarizing_error
from qiskit_aer.primitives import EstimatorV2 as AerEstimator    # Aer's own Estimator, under a name of its own

zne_noise = NoiseModel()
zne_noise.add_all_qubit_quantum_error(depolarizing_error(0.05, 2), "cx")    # every CX goes wrong 5% of the time
zne_estimator = AerEstimator(options={"backend_options": {"noise_model": zne_noise}})


def folded_bell(scale):
    """A Bell pair in which the CX is written `scale` times. `scale` must be an odd number."""
    circuit = QuantumCircuit(2)
    circuit.h(0)
    for _ in range(scale):
        circuit.cx(0, 1)
        circuit.barrier()          # a separator, so that nothing merges or cancels the repeated CX gates
    return circuit


zz = SparsePauliOp("ZZ")           # +1 when the two bits are equal, -1 when they differ
E = {}                             # E[1] and E[3]: the expectation value at the noise scales 1 and 3
for scale in (1, 3):
    zne_job = zne_estimator.run([(folded_bell(scale), zz)])
    E[scale] = float(zne_job.result()[0].data.evs)
E0 = (3 * E[1] - E[3]) / 2
print("E(1) =", round(E[1], 4), "| E(3) =", round(E[3], 4), "| extrapolated E0 =", round(E0, 4), "| ideal: 1.0")

At the normal noise level the result is 0.95, because one CX in twenty leaves a random state, whose $ZZ$ averages to 0. With every CX tripled it is 0.8574. The straight line through these two points reaches 0.9963 at zero noise. The error went from 0.05 down to 0.0037 without a single gate getting better.

It did not go to zero, because the true trend is not a straight line. Here it is the curve $0.95^{\lambda}$. A straight line from 1 through 0.95 would reach 0.85 at the scale 3, and the curve gives 0.8574, a little more. So the line through the two measured points ends slightly below 1 at zero noise. That is why real implementations fit other curves.

### Run every circuit in three places

*Slide 28.*

- **Ideal simulator.** The answer the math predicts, with no noise at all.
- **Noisy simulator.** A model of one machine, built from that machine's own calibration data.
- **The processor.** The real answer, with every error included.

The gap between the first and the third is what suppression and mitigation try to close.

These are three places to run the same circuit, and comparing them tells you where a problem comes from. If the ideal simulator gives a wrong answer, the circuit itself is wrong. If the ideal answer is right and the noisy simulator is off, noise is the cause, and you know roughly how much to expect. The processor has the last word, because no model contains every error.

In Qiskit, `AerSimulator.from_backend(backend)` builds a noisy simulator from the error rates a backend reports. The simulated processor of this notebook is such a model, built from a saved record of ibm_fez. With `USE_HARDWARE = False` it stands in for the processor in tonight's labs.

The cell below takes one circuit, the Bell pair, to all three places and prints the expectation value of $ZZ$ each time.

1. The ideal simulator, through `StatevectorEstimator`.
2. The noisy simulator. The circuit is transpiled for the simulated processor and run for 4,096 shots, and $ZZ$ is averaged over the shots.
3. A processor. The cell does not send a job. It uses the counts that were recorded when this circuit ran on the IBM processor ibm_marrakesh on Nov 13, 2026.

The cell takes a few seconds, because the model covers all 156 qubits.

In [ ]:
from qiskit import QuantumCircuit
from qiskit.primitives import StatevectorEstimator
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2

three_ways = QuantumCircuit(2)                 # the Bell pair
three_ways.h(0)
three_ways.cx(0, 1)

# 1. Ideal simulator: exact, no noise.
ideal_value = float(StatevectorEstimator().run([(three_ways, SparsePauliOp("ZZ"))]).result()[0].data.evs)

# 2. Noisy simulator: the simulated processor, a model built from the calibration data of ibm_fez.
three_ways.measure_all()
three_ways_isa = generate_preset_pass_manager(
    optimization_level=1, backend=SIMULATED_BACKEND, seed_transpiler=0).run(three_ways)
noisy_sampler = SamplerV2(mode=SIMULATED_BACKEND)
noisy_sampler.options.simulator.seed_simulator = 19        # fixes the random numbers of the simulation
noisy_bits = noisy_sampler.run([three_ways_isa], shots=4096).result()[0].data.meas
noisy_value = float(noisy_bits.expectation_values(["ZZ"])[0])
print("counts on the noisy simulator:", noisy_bits.get_counts())

# 3. A processor: counts recorded for this circuit on the IBM processor ibm_marrakesh on Nov 13, 2026.
recorded = {"00": 507, "11": 430, "01": 82, "10": 5}
recorded_value = (recorded["00"] + recorded["11"] - recorded["01"] - recorded["10"]) / sum(recorded.values())

print(f"ideal simulator: {ideal_value:.3f}")
print(f"noisy simulator: {noisy_value:.3f}")
print(f"processor:       {recorded_value:.3f}")

The ideal value is 1. On the noisy simulator 136 of the 4,096 shots gave two different bits, and the average of $ZZ$ is 0.934. On the real processor 87 of 1,024 shots did, and the average is 0.830.

Each step is further from 1: the math, then the model, then the machine. Do not read too much into the size of the last gap. The model is a copy of ibm_fez and the recorded run is from ibm_marrakesh, a different processor, and a model built from a few numbers per qubit is an approximation in any case. Lab 3 makes the same three-way comparison for a larger circuit, with and without mitigation.

### The bit-flip code spreads one qubit across three

*Slide 29.*

- $\alpha\lvert 0 \rangle + \beta\lvert 1 \rangle$ becomes $\alpha\lvert 000 \rangle + \beta\lvert 111 \rangle$.
- Two parity checks say where a flip happened without revealing $\alpha$ or $\beta$.
- It is also called the three-qubit repetition code.

An ordinary computer can protect a bit by repetition: store it three times, and if one copy is flipped, the other two outvote it. A qubit cannot be protected that way, for two reasons. An unknown quantum state cannot be copied, which is the no-cloning theorem from Wednesday's session. And comparing copies by looking at them would be a measurement, which destroys a superposition.

The bit-flip code gets around both problems. An error-correcting code is a rule for storing one qubit of information, called the logical qubit, in several physical qubits, called the data qubits. This code uses three. It stores 0 as 000 and 1 as 111, and a superposition keeps its two amplitudes: $\alpha\lvert 0 \rangle + \beta\lvert 1 \rangle$ becomes $\alpha\lvert 000 \rangle + \beta\lvert 111 \rangle$. That is one entangled state of three qubits, not three copies. The derivation below shows the difference.

The error this code handles is the bit flip: an X gate that nobody asked for, on one of the three qubits. To find it, the code asks two questions that do not reveal the stored value. Are qubits 1 and 2 equal? Are qubits 2 and 3 equal? Such a question is called a parity check. In both parts of the encoded state, 000 and 111, all three qubits are equal, so both answers are "equal" whatever $\alpha$ and $\beta$ are. A flip changes the answers, and the pattern of answers says which qubit was flipped.

Each check is measured through an extra qubit, called a check qubit. Ancilla is the general name for any extra qubit that holds none of your data. A check qubit is one kind of ancilla, and so was qubit 25, which the transpiler borrowed earlier. Two CX gates, one from each of the two data qubits onto the check qubit, leave the check qubit in 1 exactly when the two data qubits differ. Only the check qubit is measured. The data qubits are never looked at, and nothing was copied, so the no-cloning theorem is not violated.

The name "repetition code" comes from the two values 0 and 1, which are stored as a bit repeated three times.

The picture on the slide is a circuit with three data qubits and two check qubits that measure the two parities. The cell below builds that circuit as a function, so that later cells can reuse it, and draws it with a flip on the middle data qubit.

In [ ]:
import numpy as np
from qiskit import ClassicalRegister, QuantumCircuit, QuantumRegister


def bit_flip_circuit(flip=None, read_data=False):
    """The three-qubit bit-flip code: encode, flip one data qubit (0, 1, 2, or None for no flip), measure both checks."""
    data = QuantumRegister(3, "data")                # a register is a named group of qubits or of bits
    check = QuantumRegister(2, "check")
    syn = ClassicalRegister(2, "syn")                # two ordinary bits for the results of the two checks
    circuit = QuantumCircuit(data, check, syn)
    circuit.ry(2 * np.arcsin(0.8), data[0])          # the state to protect: 0.6|0> + 0.8|1> on the first data qubit
    circuit.cx(data[0], data[1])                     # encode: spread it over three qubits
    circuit.cx(data[0], data[2])
    circuit.barrier()
    if flip is not None:
        circuit.x(data[flip])                        # the error: one bit flip
        circuit.barrier()
    circuit.cx(data[0], check[0])                    # check 1 and 2: are the slide's data qubits 1 and 2 equal?
    circuit.cx(data[1], check[0])
    circuit.cx(data[1], check[1])                    # check 2 and 3: are the slide's data qubits 2 and 3 equal?
    circuit.cx(data[2], check[1])
    circuit.measure(check, syn)                      # only the two check qubits are measured
    if read_data:
        out = ClassicalRegister(3, "out")
        circuit.add_register(out)
        circuit.measure(data, out)                   # for a demonstration: read the data qubits as well
    return circuit


bit_flip_circuit(flip=1).draw("mpl", scale=0.8)

Read the drawing from left to right. Qiskit numbers from 0, so `data_0`, `data_1` and `data_2` are the slide's qubits 1, 2 and 3.

- The box $R_Y$ puts the first data qubit into the state to protect. $R_Y$ is a rotation gate: a one-qubit gate that takes an angle and can turn 0 into any blend of 0 and 1 with ordinary numbers as amplitudes. The example uses $\alpha = 0.6$ and $\beta = 0.8$, so a measurement of this qubit would give 0 with probability 0.36 and 1 with probability 0.64.
- Two CX gates encode: each has the first data qubit as its control, drawn as a dot, and another data qubit as its target, drawn as a circled plus.
- Between the dashed lines, one X gate plays the error. The dashed lines are barriers. A barrier is a separator in the drawing and does nothing to the qubits.
- Four CX gates write the two parities onto the check qubits, and the two dials at the right measure the check qubits. The double line `syn` at the bottom carries the two measured bits.

#### Derivation: Spreading is not copying

*Slide 30.*

Every line of this derivation uses the slide's order: qubit 1 is the left character.

Encode with two CX gates. Start with the first qubit in $\alpha\lvert 0 \rangle + \beta\lvert 1 \rangle$ and the other two in 0, which is $\alpha\lvert 000 \rangle + \beta\lvert 100 \rangle$. Each CX has the first qubit as its control, so it flips its target only in the part where the first qubit is 1.

$$\alpha\lvert 0 \rangle + \beta\lvert 1 \rangle \rightarrow \alpha\lvert 000 \rangle + \beta\lvert 111 \rangle$$

Three copies: a different state. Three separate qubits, each in $\alpha\lvert 0 \rangle + \beta\lvert 1 \rangle$, are the tensor product of three such states. The exponent $\otimes 3$ is shorthand for that product. Multiplied out, it has all eight terms from 000 to 111, with amplitudes such as $\alpha^3$ for 000 and $\alpha^2\beta$ for 001.

$$\left( \alpha\lvert 0 \rangle + \beta\lvert 1 \rangle \right)^{\otimes 3}$$

The no-cloning theorem forbids making this state from one unknown qubit. It allows the encoded state above, which has only two terms. The encoded state is entangled, and no single one of its qubits holds a copy of the original.

A flip on the first qubit. An X on qubit 1 changes the first character in both terms.

$$\alpha\lvert 100 \rangle + \beta\lvert 011 \rangle$$

Both parities, on either branch. A branch is one of the two terms of the state, 100 or 011. $Z_1 Z_2$ is the parity of qubits 1 and 2, written as an observable: $+1$ if the two bits are equal and $-1$ if they differ, like the $ZZ$ of the Bell pair. In the term 100, qubits 1 and 2 differ. In the term 011, they differ as well. Qubits 2 and 3 are equal in both terms.

$$Z_1 Z_2 = -1, \quad Z_2 Z_3 = +1$$

This last line is the key fact. Both terms give the same two parities. So measuring the parities says where the error is and says nothing about $\alpha$ or $\beta$. A measurement disturbs a superposition only if its result differs between the terms. Here it does not, and the superposition survives the check.

The cell below computes these four lines with `Statevector`, which holds the exact amplitudes of a state. It prints the labels in the slide's order.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp, Statevector

beta = 0.8                                # the amplitude of 1. Try any number between 0 and 1.
alpha = (1 - beta ** 2) ** 0.5            # the amplitude of 0, here 0.6


def slide_order(state):
    """The amplitudes of a three-qubit state that are not zero, labeled in the slide's order: qubit 1 on the left."""
    shown = {}
    for k, amplitude in enumerate(state.data):
        if abs(amplitude) > 1e-9:
            qiskit_label = format(k, "03b")          # the label Qiskit prints, with its qubit 0 on the right
            shown[qiskit_label[::-1]] = round(float(amplitude.real), 3)      # [::-1] reverses the label
    return dict(sorted(shown.items()))


encoder = QuantumCircuit(3)
encoder.ry(2 * np.arcsin(beta), 0)        # Qiskit's qubit 0, the slide's qubit 1, in alpha|0> + beta|1>
encoder.cx(0, 1)                          # the two CX gates of the encoding
encoder.cx(0, 2)
encoded = Statevector(encoder)
print("encoded with two CX gates:", slide_order(encoded))

one_qubit = Statevector([alpha, beta])
three_copies = one_qubit.tensor(one_qubit).tensor(one_qubit)
print("three copies would be:    ", slide_order(three_copies))

flip_first = QuantumCircuit(3)
flip_first.x(0)                           # a bit flip on the slide's qubit 1
flipped = encoded.evolve(flip_first)
print("after a flip on qubit 1:  ", slide_order(flipped))

z1z2 = SparsePauliOp.from_sparse_list([("ZZ", [0, 1], 1)], num_qubits=3)     # the parity of the slide's qubits 1 and 2
z2z3 = SparsePauliOp.from_sparse_list([("ZZ", [1, 2], 1)], num_qubits=3)     # the parity of the slide's qubits 2 and 3
print("Z1 Z2 =", round(float(flipped.expectation_value(z1z2).real), 6),
      "| Z2 Z3 =", round(float(flipped.expectation_value(z2z3).real), 6))

The encoded state has two amplitudes, 0.6 and 0.8. Three copies would have eight, starting with $0.6^3 = 0.216$. After the flip the two labels are 100 and 011, as on the slide. The two parities come out as exactly $-1$ and $+1$. They are not averages over shots that vary: every single shot gives these values. Change `beta` and run the cell again. The amplitudes change and the parities do not.

### The syndrome says where, never what

*Slide 31.*

| Error | Check 1 and 2 | Check 2 and 3 | Fix |
|---|---|---|---|
| None | 0 | 0 | Nothing |
| Flip on qubit 1 | 1 | 0 | X on qubit 1 |
| Flip on qubit 2 | 1 | 1 | X on qubit 2 |
| Flip on qubit 3 | 0 | 1 | X on qubit 3 |

- Each check is measured on its own check qubit.
- The two results together are the syndrome.
- Four syndromes, four cases: the fix is a lookup.
- Qubit 1 is $\text{data}_0$ in the circuit and `d[0]` in the code.

A check gives 0 when its two data qubits are equal and 1 when they differ. These are the parities $+1$ and $-1$ of the derivation. The two results together are called the syndrome, a word borrowed from medicine: the signs that point to what is wrong.

With at most one flip there are four cases, and each has its own syndrome. A flip on qubit 2 is the only one that disturbs both checks, because qubit 2 takes part in both. The repair is a lookup in the table: apply X to the qubit the syndrome points to. Two X gates on the same qubit cancel, so the second one undoes the error.

The title sums it up. The syndrome says where the flip happened. It never says what state is stored.

The cell below runs the circuit drawn above for the four cases, 1,000 shots each. This time it also measures the three data qubits at the end, to show what happened to them. The data column is printed in the slide's order, qubit 1 on the left.

In [ ]:
from qiskit_aer.primitives import SamplerV2 as AerSampler

syndrome_sampler = AerSampler(seed=22)                     # the seed fixes the random numbers
print("Error             Check 1 and 2   Check 2 and 3   Data qubits 1, 2, 3 as read")
for label, flip in [("None", None), ("Flip on qubit 1", 0), ("Flip on qubit 2", 1), ("Flip on qubit 3", 2)]:
    result = syndrome_sampler.run([bit_flip_circuit(flip, read_data=True)], shots=1000).result()[0]
    syndromes = result.data.syn.get_counts()               # the syndromes of the 1,000 shots, counted
    printed = list(syndromes)[0]                           # Qiskit prints check 2 and 3 first, then check 1 and 2
    data_read = {}
    for bits, shots in result.data.out.get_counts().items():
        data_read[bits[::-1]] = shots                      # [::-1] reverses a string: qubit 1 on the left
    print(f"{label:17s} {printed[1]:15s} {printed[0]:15s} {dict(sorted(data_read.items()))}"
          f"   different syndromes seen: {len(syndromes)}")

The two check columns are the table of the slide, row for row, and in every row all 1,000 shots gave the same syndrome.

The data column shows the damage. With a flip on qubit 2 the data reads 010 or 101, never 000 or 111. It also shows that the syndrome is blind to the stored value. Within one row about 36% of the shots read one pattern and about 64% read the other, which are the probabilities $0.6^2$ and $0.8^2$, and the syndrome is the same for both. The split is the same in every row, give or take one shot, because each row is run with the same seed and so with the same random numbers.

One detail about order. Qiskit prints the two syndrome bits with the later bit first, so check 2 and 3 comes before check 1 and 2. The cell rearranges them into the slide's order. The "Try it" cell in Part 2 prints them in Qiskit's order.

Slide 32 is a cell to try. It measures the syndrome in the middle of the circuit and lets the result choose the fix. You find it at the start of Part 2, under "Try it: fix a flip while the circuit is still running".

### Three noisy bits beat one, when the noise is small

*Slide 33.*

$$p_L = 3p^2\left( 1 - p \right) + p^3 = 3p^2 - 2p^3$$

At $p = 1\%$, the encoded bit fails 0.03% of the time, about 34 times better.

The code's limit: it misses phase flips, where the sign between the 0 part and the 1 part flips.

$p$ is the probability that one physical qubit gets flipped. Assume that each of the three data qubits is flipped with this probability, independently of the others. The code repairs any single flip. Two flips are too many: the syndrome then points to the wrong qubit, and the "repair" completes the damage. For example, flips on qubits 1 and 2 turn 000 into 110. The syndrome is 0 and 1, the same as for a single flip on qubit 3, so the fix is an X on qubit 3, and the result is 111. The stored bit is now wrong. That is called a logical error, and $p_L$ is its probability, the logical error rate.

Another way to say the same thing: the code takes a majority vote. It returns the value that at least two of the three qubits show, so it fails when two or three of them are flipped.

At $p = 0.01$ the formula gives $p_L = 0.000298$, about 0.03%, and $0.01 / 0.000298 = 33.6$. The encoded bit is about 34 times more reliable than a bare one.

The code has two limits. It helps only when $p$ is small enough, and the cell below shows where the gain ends. And it catches bit flips only. A phase flip, an unwanted Z gate, turns $\alpha\lvert 000 \rangle + \beta\lvert 111 \rangle$ into $\alpha\lvert 000 \rangle - \beta\lvert 111 \rangle$. Both parity checks still answer "equal", so it passes unnoticed. A real code has to check for both kinds of error.

#### Derivation: The logical error rate, term by term

*Slide 35.*

Exactly two flips: three ways. The two flipped qubits can be 1 and 2, 1 and 3, or 2 and 3. In each case two qubits are flipped, with probability $p$ each, and the third is not, with probability $1 - p$.

$$3p^2(1 - p)$$

All three flip.

$$p^3$$

Total. The majority vote fails in these two cases and in no other. Multiplying out, $3p^2(1 - p) = 3p^2 - 3p^3$.

$$p_L = 3p^2(1 - p) + p^3 = 3p^2 - 2p^3$$

At $p = 0.01$.

$$3 \times 10^{-4} - 2 \times 10^{-6} = 2.98 \times 10^{-4}$$

The cell below checks the formula by brute force. For two values of $p$ it flips three bits at random, a million times, and counts how often two or more of them were flipped. Then it draws $p_L$ against $p$ for every $p$ from 0 to 1.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

rng = np.random.default_rng(2026)                # the seed fixes the random numbers
trials = 1_000_000
for p in (0.01, 0.1):
    flips = rng.random((trials, 3)) < p          # in each trial, each of the 3 qubits flips with probability p
    failed = flips.sum(axis=1) >= 2              # the majority vote fails when 2 or 3 of them flipped
    formula = 3 * p ** 2 - 2 * p ** 3
    print(f"p = {p}: failed in {failed.mean():.6f} of the trials | formula {formula:.6f} | gain: {p / formula:.1f} times")

p_values = np.linspace(0, 1, 201)
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(p_values, p_values, color="0.5", ls="--", label="one bare qubit: p")
ax.plot(p_values, 3 * p_values ** 2 - 2 * p_values ** 3, color="tab:blue", label="three qubits and a majority vote")
ax.plot(0.5, 0.5, "o", color="black")
ax.annotate("p = 0.5: no gain", (0.5, 0.5), textcoords="offset points", xytext=(8, -12))
ax.set_xlabel("chance p that one qubit is flipped")
ax.set_ylabel("chance that the stored bit is wrong")
ax.set_title("Three noisy bits against one")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

At $p = 0.01$ the vote failed in 299 of the million trials, against 298 predicted by the formula. At $p = 0.1$ it failed in 2.8% of the trials. That is still better than 10%, but only by a factor of 3.6.

In the figure the solid curve lies below the dashed line for every $p$ below one half. At one half the two meet, and above it the code makes things worse. The code helps for any $p$ below one half, and it helps most when $p$ is small.

### The surface code checks both, in two dimensions

*Slide 36.*

- A $d \times d$ patch of data qubits with check qubits between them.
- Two kinds of check, pink and blue: one kind finds bit flips, the other phase flips.
- Distance $d = 3$ uses 17 qubits. Distance $d = 5$ uses 49.
- Bigger patches fail less, as long as physical error is under about 1%.

The surface code is the idea of the bit-flip code, grown in two directions. Its data qubits form a square grid, called a patch, with $d$ qubits on a side. Between them sit check qubits. Each check qubit measures a parity of the data qubits around it, again and again while the computation runs. There are two kinds of check. One kind finds bit flips, like the checks of the bit-flip code. The other kind finds phase flips. "Checks both" in the title means both kinds of error.

The number $d$ is called the distance of the code. It is the smallest number of single-qubit errors that can change the stored value without any check noticing. The bit-flip code has distance 3 against bit flips: it takes three flips to turn 000 into 111. A code of distance $d$ can be fooled by as few as $\frac{d+1}{2}$ errors at the same time, if they fall in the wrong places. That is 2 errors for $d = 3$, as you saw with the majority vote, and 3 errors for $d = 5$. Fewer errors than that are always found and repaired.

The last bullet describes a threshold. A bigger patch tolerates more errors, but it also has more qubits and more gates in which errors can happen. Which effect wins depends on the physical error rate, the chance that a single operation goes wrong. Below a certain value, called the threshold and written $p_{\text{th}}$, a bigger patch fails less often. Above it, a bigger patch fails more often. For the surface code the threshold is about 1%.

The picture on the slide is a distance-3 patch: nine data qubits as circles and eight check qubits as squares. The cell below draws it. On the slide the two kinds of check are pink and blue; in the drawing below they are orange and green.

In [ ]:
import matplotlib.pyplot as plt

patch = 3                                            # the distance d of the patch
data_spots = [(x, y) for x in range(patch) for y in range(patch)]

# Check qubits sit in the middle of the squares between data qubits. The two kinds alternate like the
# colors of a chessboard. Along the edges only every other square holds a check, and it touches two data qubits.
check_spots = []                                     # one entry (x, y, kind) for each check qubit
for i in range(-1, patch):
    for j in range(-1, patch):
        kind = "bit" if (i + j) % 2 == 0 else "phase"
        inside = 0 <= i < patch - 1 and 0 <= j < patch - 1
        top_or_bottom = 0 <= i < patch - 1 and j in (-1, patch - 1) and kind == "phase"
        left_or_right = 0 <= j < patch - 1 and i in (-1, patch - 1) and kind == "bit"
        if inside or top_or_bottom or left_or_right:
            check_spots.append((i + 0.5, j + 0.5, kind))

fig, ax = plt.subplots(figsize=(4.6, 4.6))
colors = {"bit": "tab:orange", "phase": "tab:green"}
for check_x, check_y, kind in check_spots:           # a line from each check qubit to the data qubits it checks
    for x, y in data_spots:
        if abs(x - check_x) == 0.5 and abs(y - check_y) == 0.5:
            ax.plot([check_x, x], [check_y, y], color=colors[kind], lw=1.5, zorder=1)
for kind, label in [("bit", "check qubit: finds bit flips"), ("phase", "check qubit: finds phase flips")]:
    spots = [(check_x, check_y) for check_x, check_y, spot_kind in check_spots if spot_kind == kind]
    ax.scatter([spot[0] for spot in spots], [spot[1] for spot in spots], marker="s", s=170,
               color=colors[kind], zorder=2, label=label)
ax.scatter([x for x, y in data_spots], [y for x, y in data_spots], marker="o", s=230, color="#1f4e9c",
           zorder=3, label="data qubit")
ax.set_aspect("equal")
ax.axis("off")
ax.set_title("A distance-3 surface code patch")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.02), frameon=False)
plt.show()

print("data qubits:", len(data_spots), "| check qubits:", len(check_spots),
      "| total:", len(data_spots) + len(check_spots))

The nine circles are the data qubits. Each square is a check qubit, joined by lines to the data qubits it checks. A check qubit in the middle of the patch checks four data qubits, and a check qubit on the edge checks two. The two colors are the two kinds of check, four of each. Together that is 17 qubits to store one logical qubit.

#### Derivation: How the surface code scales

*Slide 37.*

Qubits at distance $d$. A patch has $d \times d$ data qubits and one check qubit fewer.

$$d^2 + (d^2 - 1) = 2d^2 - 1$$

For $d = 3, 5, 7$.

$$17, \quad 49, \quad 97$$

Logical error below threshold scales like the formula below. $p$ is the physical error rate and $p_{\text{th}}$ is the threshold. The symbol $\propto$ means "is proportional to": equal up to a constant factor. The exponent is the smallest number of errors that can fool the code.

$$p_L \propto \left( \frac{p}{p_{\text{th}}} \right)^{(d+1)/2}$$

With $p$ ten times below threshold. Then $\frac{p}{p_{\text{th}}} = 0.1$, and the exponents for $d = 3, 5, 7$ are 2, 3 and 4.

$$d = 3: \; 10^{-2}, \quad d = 5: \; 10^{-3}, \quad d = 7: \; 10^{-4}$$

The last line leaves out the constant in front, so read it as a trend and not as error rates: every two steps of distance buy another factor of ten. The cell below prints both sides of the trade, the qubits you pay and the factor you gain.

In [ ]:
print("distance d | data qubits | check qubits | total | (p / p_th)^((d + 1) / 2) at p / p_th = 0.1")
for distance in (3, 5, 7):
    data_count = distance ** 2
    check_count = distance ** 2 - 1
    factor = 0.1 ** ((distance + 1) / 2)
    print(f"     {distance}     |     {data_count:2d}      |      {check_count:2d}      |  {data_count + check_count:2d}   | {factor:.0e}")

From distance 3 to distance 7 the factor falls from `1e-02` to `1e-04`, which is how Python writes $10^{-2}$ and $10^{-4}$. The number of physical qubits for one logical qubit grows from 17 to 97.

### What a fault-tolerant processor needs

*Slide 38.*

- **Errors below threshold.** Physical error rates low enough that a bigger code helps more than it hurts.
- **Many qubits per logical qubit.** A distance-$d$ surface code patch uses $2d^2 - 1$ physical qubits.
- **Decoding in real time.** Syndromes are read every cycle and must be decoded as fast as they come in.
- **Logical gates.** A full set of gates that act on encoded qubits without spreading errors.

The check qubits, and the gates that carry out the checks, are as noisy as everything else on the chip. Fault tolerance means that the error-correction machinery is itself built so that one fault cannot spread into many. A processor that computes this way needs the four things on the cards.

- The first card is the threshold of the surface code slide. Above it, adding qubits makes things worse.
- The second card is the count from the derivation: 17, 49, 97 and so on, for every logical qubit.
- In the third card, a cycle is one round of all the checks. A decoder is the ordinary program that turns the stream of syndromes into a decision about which errors happened. For the bit-flip code it was a lookup in a table with four rows. For a large patch it is a much bigger computation, and it has to keep up with the processor.
- The fourth card: storing a qubit safely is not yet computing. The gates of an algorithm have to act on the encoded qubits, and they must do so without turning one error into several.

Suppression and mitigation, the first two rows of the table earlier tonight, work on the processors you can use today. Fault-tolerant correction needs all four cards at once. The "Try it" cell in Part 2 runs the smallest example of correction, the bit-flip code, on a simulator.

## Part 2: Hands-on

The code in the "Try it" and "Lab" cells is the code on the slides. "File on the slide" gives the name shown above that code on the slide; there is no separate file to download. Run each cell, read what it prints, then change something and run it again. After each lab comes a "Your turn" exercise with a check cell. Part 2 needs only the Setup section, so you can start here.

### Try it: fix a flip while the circuit is still running

*Slide 32.*

File on the slide: `bitflip_feedforward.py`

1. Encode a logical 1 across three qubits
2. Flip one qubit on purpose
3. Measure both parity checks mid-circuit
4. Let the result choose the fix: feed-forward
5. Read the data: 111 on every shot

This is the bit-flip code from Part 1 with one new step: the repair happens inside the circuit. The two check qubits are measured in the middle of the circuit, which is called a mid-circuit measurement, and the measured bits decide which gate runs next. Using a measurement result to choose later gates of the same circuit is called feed-forward. A circuit that does this is called a dynamic circuit.

Line by line:

- The four registers. `d` holds the three data qubits and `a` the two check qubits. `s` holds two ordinary bits for the syndrome and `m` three ordinary bits for the final readout.
- `qc.x(d[0]); qc.cx(d[0], d[1]); qc.cx(d[0], d[2])` is step 1. The X makes a 1 and the two CX gates spread it over the three data qubits, which gives 111, a logical 1. A semicolon puts several statements on one line.
- `qc.x(d[1])` is step 2, the error. `d[1]` is the slide's qubit 2.
- The next two lines write the two parities onto the check qubits, and `qc.measure(a, s)` measures them into `s`. That is step 3.
- The three `with qc.if_test(...)` lines are step 4. Each one reads: if the syndrome register holds this number, apply this X gate. The register `s` is read as a number in which check 1 and 2 counts 1 and check 2 and 3 counts 2. So 1 means that only check 1 and 2 gave a 1, which points to qubit 1. 3 means that both checks did, which points to qubit 2. 2 means that only check 2 and 3 did, which points to qubit 3. These three lines are the lookup table of the slide.
- `qc.measure(d, m)` is step 5.
- `SamplerV2` here is the Sampler of Aer, the simulator. Unlike the ideal `StatevectorSampler`, it can run a circuit with a mid-circuit measurement and `if_test`.

In [ ]:
from qiskit import (QuantumCircuit,
    QuantumRegister, ClassicalRegister)
from qiskit_aer.primitives import SamplerV2

d = QuantumRegister(3, "data")
a = QuantumRegister(2, "check")
s = ClassicalRegister(2, "syn")
m = ClassicalRegister(3, "out")
qc = QuantumCircuit(d, a, s, m)
qc.x(d[0]); qc.cx(d[0], d[1]); qc.cx(d[0], d[2])
qc.x(d[1])                     # the error
qc.cx(d[0], a[0]); qc.cx(d[1], a[0])
qc.cx(d[1], a[1]); qc.cx(d[2], a[1])
qc.measure(a, s)               # mid-circuit
with qc.if_test((s, 1)): qc.x(d[0])
with qc.if_test((s, 3)): qc.x(d[1])
with qc.if_test((s, 2)): qc.x(d[2])
qc.measure(d, m)
res = SamplerV2().run([qc], shots=200).result()[0]
print(res.data.syn.get_counts(),
      res.data.out.get_counts())

The cell prints `{'11': 200} {'111': 200}`. The first dictionary is the syndrome: all 200 shots gave `11`. Both checks gave a 1, which the table reads as a flip on qubit 2. The second dictionary is the data: `111` on every shot. The flip was found and repaired while the circuit was still running.

The syndrome string is read from right to left: its right-most character is the check on qubits 1 and 2. With `11` you cannot see that, so change the error. Replace `qc.x(d[1])` by `qc.x(d[0])` and run the cell again. It prints the syndrome `01`: check 1 and 2 gave 1 and check 2 and 3 gave 0, the table's row for qubit 1. With `qc.x(d[2])` it prints `10`, and with the error line removed it prints `00`. In all four cases the data reads `111`.

This cell runs on the simulator in every setting. On real hardware `if_test` needs a backend that supports dynamic circuits.

One housekeeping cell follows. The cell above imported Aer's `SamplerV2`. The package qiskit-ibm-runtime has a class with the same name, the one this week's hardware cells use as `SamplerV2(mode=backend)`. The next cell points the name back at that class, so that such a line still works after the "Try it" cell. Without it, the line stops with a `TypeError` about an unexpected keyword argument `mode`.

In [ ]:
# Glue: the cell above imported Aer's SamplerV2. Point the names SamplerV2 and EstimatorV2
# at the classes of qiskit-ibm-runtime again, the ones that accept mode=backend.
from qiskit_ibm_runtime import SamplerV2, EstimatorV2

### Lab 1: read the machine's own report card

*Slide 43.*

File on the slide: `report_card.py`

1. Pick a backend
2. Pull $T_1$, $T_2$, and readout error
3. Pull the two-qubit gate errors
4. Find the best and worst qubit pairs

Step 1 is the small cell below. `connect("ibm")` returns the service, as described in the Setup section, and `least_busy` picks a backend. With `USE_HARDWARE = False` that is the simulated processor. With your own account it is the working IBM processor with the fewest jobs waiting, and the calibration data in this lab is then the live record of that machine. The cell prints the name of the backend and its number of qubits.

This notebook has no cell marked "second option" until after Lab 3. Labs 1 to 3 read the calibration data of an IBM processor and use options of Qiskit Runtime. With `HARDWARE = "qbraid"` their cells run on the simulated processor.

**Hardware cell (IBM Quantum).** With `USE_HARDWARE = True` and `HARDWARE = "ibm"` this cell uses your own IBM Quantum account. With any other setting it runs on a simulated IBM processor and uses no account. Lab 1 only reads the calibration record and uses no quantum time, even though the cell's message mentions it.

In [ ]:
# Glue for Lab 1: step 1 of the slide. The lab code below expects a backend.
service = connect("ibm")
backend = service.least_busy(
    operational=True, simulator=False)
print(backend.name, backend.num_qubits)

The lab cell reads the report card. Line by line:

- `backend.properties()` returns the calibration record of the qubits. `props.t1(q)` and `props.t2(q)` are the $T_1$ and $T_2$ of qubit `q` in seconds, and `props.readout_error(q)` is its readout error, a probability. The loop prints one line for each of the qubits 0 to 4: the qubit, $T_1$, $T_2$, readout error.
- `backend.target` describes everything the backend can do: each operation, on each qubit or pair, with its error and its duration. `target.operation_names` lists the names of the operations.
- `gate = "cz" if "cz" in names else "ecr"` finds the name of the two-qubit gate. It depends on the processor family, so the code checks which one this backend reports. It is `cz` on the simulated processor and on the Open Plan processors that were checked for this notebook on Nov 13, 2026. For any other backend the code assumes the name `ecr`, the echoed cross-resonance gate, which is the two-qubit gate of IBM's earlier processors.
- `errs` is a dictionary that maps each coupled pair to the error of the two-qubit gate on that pair. Pairs without a reported error are skipped.
- `best` and `worst` are the pairs with the smallest and the largest error. The last two lines print them.

In [ ]:
props = backend.properties()
for q in range(5):
    print(q, props.t1(q), props.t2(q),
          props.readout_error(q))

target = backend.target
names = target.operation_names
gate = "cz" if "cz" in names else "ecr"

errs = {pair: p.error
        for pair, p in target[gate].items()
        if p is not None and p.error}
best = min(errs, key=errs.get)
worst = max(errs, key=errs.get)
print(gate, best, errs[best])
print(gate, worst, errs[worst])

How to read the output. Python writes small numbers with an exponent: the `4.880620782230333e-05` in the first line is $4.88 \times 10^{-5}$ seconds, which is 48.8 microseconds.

On the simulated processor, qubit 0 has the shortest $T_1$ of the five, 48.8 microseconds, and qubit 2 the longest, 274 microseconds. The readout errors run from 0.0015 on qubit 4 to 0.016 on qubit 3, a factor of ten between qubits that sit side by side. That is the first row of the "What you should see" table near the end of this notebook: the numbers differ a lot across one chip. With your own account the numbers are those of the real processor on the day you run the cell.

The best pair on the simulated processor is (22, 23), with an error of 0.0021. The worst prints as `cz (27, 28) 1.0`. An error of 1 would mean that the gate fails every time. It is not a measured rate. It is how this calibration record marks a pair whose two-qubit gate was out of use when the record was made. The next cell lists the pairs with this mark and finds the worst pair with a measured error.

In [ ]:
out_of_use = sorted({tuple(sorted(pair)) for pair, error in errs.items() if error >= 1})
in_use = {pair: error for pair, error in errs.items() if error < 1}
worst_in_use = max(in_use, key=in_use.get)
print("pairs reported with error 1:", out_of_use)
print("worst pair with a measured error:", worst_in_use, in_use[worst_in_use])
print("worst measured error divided by best error:", round(in_use[worst_in_use] / errs[best], 1))

On the simulated processor seven pairs carry the mark, and the worst pair with a measured error is (153, 154) at 0.050, about 23 times the error of the best pair. This is the information Challenge 1, Biggest GHZ state, needs: a circuit does better on the pairs with small errors, and the transpiler in Lab 2 uses the same record to choose them.

#### Your turn 1

Complete the function `survival`. It takes a waiting time `t` and a lifetime `t1`, both in seconds, and returns the fraction of a 1 that is left after that time. Use the decay law from Part 1. `np.exp(x)` computes $e^x$.

In [ ]:
import numpy as np


def survival(t, t1):
    """Return the fraction of a 1 that is left after the time t, for a qubit with the lifetime t1 (both in seconds)."""
    frac = None
    # YOUR CODE HERE
    return frac

In [ ]:
assert survival(1e-6, 1e-4) is not None, "survival() still returns None. Replace the line YOUR CODE HERE with code that sets frac."
assert isinstance(survival(1e-6, 1e-4), (int, float)), "survival() must return one number, not text and not a list."
assert abs(survival(20e-6, 200e-6) - 0.904837) < 1e-5, "The worked example of the seminar, 20 microseconds with T1 = 200 microseconds, should leave 0.905. Check the decay law in Part 1, and its sign. Return the fraction itself: no rounding, no percent."
assert abs(survival(50e-6, 100e-6) - 0.606531) < 1e-5, "survival() must use its own argument t1, not a T1 defined earlier in the notebook. With t1 = 100 microseconds, 50 microseconds should leave 0.607."
assert abs(survival(40e-6, 200e-6) - survival(20e-6, 200e-6) ** 2) < 1e-9, "Waiting twice as long should square the surviving fraction. The decay is exponential."
assert abs(survival(0.0, 1e-4) - 1) < 1e-12, "At t = 0 nothing has decayed, so the fraction must be 1."
print("Check passed.")
print("Fraction of a 1 that survives a circuit of 20 microseconds, for the qubits 0 to 4 of the backend:")
for q in range(5):
    print(q, round(float(survival(20e-6, props.t1(q))), 3))

### Lab 2: one circuit, four transpilations per seed

*Slide 44.*

File on the slide: `golf.py`

1. Transpile at optimization levels 0 to 3
2. Count two-qubit gates each time
3. Try other seeds
4. Keep the best

The code on the slide transpiles a circuit named `qc`, which the slide does not define. The small cell below defines it: a GHZ state on eight qubits, built as a star. One H goes on qubit 0, then a CX from qubit 0 to each of the other seven. It is the five-qubit star of Part 1, grown to eight qubits. The lab code also uses `backend`, which the small cell above the Lab 1 cell picks. If you start here, for example after a restart, run the Setup section and that cell first.

Why a star? Qubit 0 has to interact with seven partners, and a physical qubit has at most three neighbors. No layout can avoid routing, and how much routing is needed depends on the choices the transpiler makes. That makes the search worth it. A GHZ state built as a chain, where each qubit interacts only with the next one, fits the rows of the processor as it is, and every transpilation of it gives the same count, seven.

The circuit has no measurements, because Lab 3 needs it that way: the Estimator adds the measurements itself. (If you shrink the star to two qubits, keep it without measurements too. `measure_all()` adds a barrier across all the qubits, and on two qubits the counter on the slide, which counts every instruction that acts on exactly two qubits, would count that barrier as a gate.)

In [ ]:
# Glue for Lab 2: the circuit the slide's code calls qc. Without this cell, qc would still be
# the bit-flip circuit of the "Try it" cell, and the lab code would transpile that one without complaint.
from qiskit import QuantumCircuit

width = 8
qc = QuantumCircuit(width)        # a GHZ state built as a star: qubit 0 talks to every other qubit
qc.h(0)
for other in range(1, width):
    qc.cx(0, other)

The lab cell, line by line:

- `two_qubit_gates(circ)` counts the instructions of a circuit that act on two qubits. `circ.data` is the list of instructions, and `inst.operation.num_qubits` says on how many qubits one of them acts.
- The two loops go through the optimization levels 0 to 3 and, for each level, through the seeds 0 to 9. That makes 40 transpilations. Each one builds a pass manager for the backend, transpiles `qc` and counts.
- `best` keeps the result with the smallest count so far, as four entries: the count, the level, the seed and the transpiled circuit. The last line prints the first three.

One name in this cell was used before. In Lab 1, `best` was a pair of qubits. From here on it is these four entries, and Lab 3 takes its circuit from `best[3]`. So do not run Lab 1 between Lab 2 and Lab 3. If you go back and run Lab 1 again, run Lab 2 and then the small cell above the Lab 3 cell again before Lab 3.

In [ ]:
from qiskit.transpiler import (
    generate_preset_pass_manager)

def two_qubit_gates(circ):
    return sum(1 for inst in circ.data
               if inst.operation.num_qubits == 2)

best = None
for level in range(4):
    for seed in range(10):
        pm = generate_preset_pass_manager(
            optimization_level=level,
            backend=backend,
            seed_transpiler=seed)
        isa = pm.run(qc)
        n = two_qubit_gates(isa)
        if best is None or n < best[0]:
            best = (n, level, seed, isa)
print(best[:3])

On the simulated processor the cell prints `(15, 2, 0)`: the best of the 40 transpilations has 15 two-qubit gates, and it was found at optimization level 2 with seed 0. The same search on the IBM processor ibm_fez on Nov 13, 2026 gave the same three numbers. Seven of the 15 gates do the work of your seven CX gates. The other eight are the price of routing. By the plain count of the slides the transpiler needed four SWAPs, 12 gates, but it merged some of them with CX gates next to them on the same pair, and each merge saves two gates.

The cell prints only the winner. The next cell repeats the 40 transpilations and prints every count, so that you can see how much they differ.

In [ ]:
print("level | two-qubit gates for the seeds 0 to 9  | smallest")
all_counts = []
for level in range(4):
    row = []
    for seed in range(10):
        trial_pm = generate_preset_pass_manager(
            optimization_level=level, backend=backend, seed_transpiler=seed)
        row.append(two_qubit_gates(trial_pm.run(qc)))
    all_counts.extend(row)
    print(f"  {level}   | " + " ".join(f"{count:3d}" for count in row) + f" |   {min(row)}")
print("different counts:", sorted(set(all_counts)))
print("estimated fidelity (1 - eps)^N at eps = 0.005: fewest gates", round(0.995 ** min(all_counts), 3),
      "| most gates", round(0.995 ** max(all_counts), 3))

This is the second row of the "What you should see" table: different counts for different levels and seeds. On the simulated processor six different counts appear, from 15 to 31.

- Level 0 does no optimization, and it never wins: its counts run from 25 to 31.
- Level 1 gives 19 for every seed.
- Levels 2 and 3 give 15, 17 or 19, depending on the seed.

The same circuit on the same processor comes out with 15 two-qubit gates or with 31. By the estimate of Part 1 that is the difference between keeping 0.928 and keeping 0.856 of the result, before anything else goes wrong.

The next cell shows where the winner sits. It lists the physical qubits that the best circuit uses and marks them on the qubit map.

In [ ]:
used = sorted({best[3].find_bit(q).index for inst in best[3].data for q in inst.qubits})
print("physical qubits used by the best circuit:", used)
same_map = set(backend.coupling_map.get_edges()) == set(SIMULATED_BACKEND.coupling_map.get_edges())
if "draw_qubit_map" not in globals():          # the drawing function is defined in Part 1
    print("To see them on the qubit map, run the cell under 'Inside an IBM quantum processor' in Part 1,")
    print("then run this cell again.")
elif same_map:
    draw_qubit_map(backend, chosen=used, title="Where the best circuit sits")
else:
    print("This backend has a different qubit map, so the drawing is skipped.")

On the simulated processor the eight qubits are 140 to 145 in the bottom row, qubit 136 above qubit 143, and qubit 123 above that. They form one compact group, and the circuit still needs eight extra gates, because a star does not fit a layout in which no qubit has more than three neighbors.

#### Your turn 2

The lab tried ten seeds at each level. Search further: try the seeds 0 to 29 at the level that won, which is `best[1]`, and keep the result with the fewest two-qubit gates. Store it in `my_best`, with the same four entries as `best`: the count, the level, the seed and the transpiled circuit. Your result must be at least as good as the lab's, so start from `best` and replace it only when a seed does better. Also keep the count of every seed you try in the list `my_counts`, in the order of the seeds, so that the check can see your search. You have everything you need in the lab cell: `generate_preset_pass_manager`, `two_qubit_gates`, `backend` and `qc`.

In [ ]:
my_best = None
my_counts = []      # the two-qubit gate count for each seed, in the order of the seeds
# YOUR CODE HERE

In [ ]:
import numbers

from qiskit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager

assert my_best is not None, "my_best is still None. Start from the lab's best, then try more seeds at the level best[1]."
assert isinstance(my_best, (tuple, list)) and len(my_best) == 4, "my_best needs the same four entries as best: (count, level, seed, circuit)."
assert isinstance(my_best[3], QuantumCircuit), "The last entry of my_best must be the transpiled circuit."
assert my_best[0] == two_qubit_gates(my_best[3]), "The first entry must be the count of the circuit in the last entry. Count it with two_qubit_gates()."
allowed = set(backend.target.operation_names) | {"barrier"}
assert set(my_best[3].count_ops()) <= allowed, "The circuit in my_best contains gates this backend does not have. Run qc through a pass manager and keep what comes out."
assert my_best[3].num_qubits == backend.num_qubits, "The circuit in my_best is not laid out on this backend. Build the pass manager with backend=backend, as the lab cell does."
assert my_best[0] >= two_qubit_gates(qc), "The circuit in my_best has fewer two-qubit gates than qc itself. Transpile qc, the star, and keep what comes out."
assert my_best[1] == best[1], "The second entry must be the level that won the lab, best[1]. Search at that level and store it."
assert isinstance(my_best[2], numbers.Integral), "The third entry must be the seed you transpiled with, a whole number."
assert two_qubit_gates(generate_preset_pass_manager(optimization_level=my_best[1], backend=backend, seed_transpiler=my_best[2]).run(qc)) == my_best[0], "The level and the seed in my_best do not give this count. Store the level and the seed you transpiled with."
assert my_best[0] <= best[0], "This is worse than the lab's best. Keep the lab's result unless a new seed beats it."
assert isinstance(my_counts, (list, tuple)) and len(my_counts) == 30, "my_counts needs one count for each of the seeds 0 to 29."
for spot in (1, 13, 27, 29):          # four of the thirty seeds are counted again here
    again = two_qubit_gates(generate_preset_pass_manager(optimization_level=best[1], backend=backend, seed_transpiler=spot).run(qc))
    assert my_counts[spot] == again, f"my_counts[{spot}] should be the count for the seed {spot} at the level best[1]. Count each transpiled circuit with two_qubit_gates(), in the order of the seeds."
assert my_best[0] == min([best[0]] + list(my_counts)), "my_best is not the best of the lab's result and your 30 counts. Replace it whenever a seed gives fewer gates."
print("Check passed. Your best:", my_best[:3], "| the lab's best:", best[:3])
if my_best[0] == best[0]:
    print("Your best equals the lab's. That is a result too: none of the seeds you tried did better.")

### Lab 3: run it raw, then mitigated

*Slide 45.*

File on the slide: `mitigate.py`

1. Submit the best transpilation without mitigation
2. Submit again with mitigation on
3. Compare both to the ideal value

The circuit is the best transpilation from Lab 2. The quantity to measure is the parity of all eight qubits: $+1$ for a shot with an even number of 1s and $-1$ for a shot with an odd number. It is the $ZZ$ of Part 1 stretched to eight qubits, and in Qiskit it is written `SparsePauliOp("ZZZZZZZZ")`. An ideal GHZ state gives 00000000 or 11111111. That is zero 1s or eight 1s, an even number both times, so the ideal expectation value is exactly 1. Every bit that an error flips changes the parity of its shot, which makes this a sensitive test.

The slide's code uses two names that it does not define: `obs`, the observable, and `ideal`, the ideal value. The glue cell below defines them.

On the simulated processor the glue cell does one more thing: it swaps the backend for a lighter model. On the tested version, the Estimator of qiskit-ibm-runtime measures every qubit of the transpiled circuit, all 156 of the chip, and the full model of the simulated processor then asks for more memory than a laptop has. The lighter model keeps the 156 qubits but carries errors only for the eight qubits your circuit uses. Their numbers are copied from the calibration record: the readout error of each qubit, the error of its one-qubit gates, and the error of the two-qubit gate on each coupled pair among them. The model leaves out $T_1$ and $T_2$ decay. It is simulated with the matrix-product-state method from Thursday's session. That method stores the state as a chain of small tables of numbers, one for each qubit, and the tables stay small as long as there is little entanglement between the qubits. It copes with the 156 measured qubits because all but eight of them stay in 0, and it keeps working if you change the circuit. From this cell on, the name `backend` points at the lighter model, so the Lab 3 cell can be run as often as you like. To get the full simulated processor back, run the small cell above the Lab 1 cell again. With your own account nothing is swapped, and the jobs go to the real processor.

**Hardware cell (IBM Quantum).** This cell prepares Lab 3 and sends nothing. With `USE_HARDWARE = True` and `HARDWARE = "ibm"` it leaves `backend` as your real IBM processor. With any other setting it builds the lighter model of the simulated processor.

In [ ]:
# Glue for Lab 3: the observable and the ideal value that the slide's code expects.
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error

obs = SparsePauliOp("Z" * width)   # the parity of all eight qubits
ideal = 1.0                        # a GHZ state on an even number of qubits has parity +1

if type(backend).__name__ == "SimulatedBackend":
    full = backend                 # keep the simulated processor under a second name
if type(backend).__name__ in ("SimulatedBackend", "AerSimulator"):
    # Simulated processor only: copy the error rates of the qubits this circuit uses
    # into a small noise model that a laptop can simulate. Running this cell again builds the model again.
    record = full.target           # its calibration record, as in Lab 1
    used = sorted({best[3].find_bit(q).index
                   for inst in best[3].data for q in inst.qubits})
    noise = NoiseModel(basis_gates=list(record.operation_names))
    for q in used:
        ro = record["measure"][(q,)].error                      # the readout error of this qubit
        noise.add_readout_error([[1 - ro, ro], [ro, 1 - ro]], [q])
        noise.add_quantum_error(                                # the error of its one-qubit gates
            depolarizing_error(record["sx"][(q,)].error, 1), ["sx", "x"], [q])
    for pair, prop in record["cz"].items():
        if set(pair) <= set(used):                              # the two-qubit gates among these qubits
            noise.add_quantum_error(depolarizing_error(prop.error, 2), "cz", pair)
    backend = AerSimulator.from_backend(full, noise_model=noise, method="matrix_product_state")
    print("Simulated processor: Lab 3 runs on a lighter model, with errors on the qubits", used)

The lab cell, line by line:

- `isa = best[3]` is the best circuit from Lab 2.
- `obs_isa = obs.apply_layout(isa.layout)`. The observable `obs` is written for your eight qubits. The transpiled circuit has 156 qubits, and your eight sit on particular physical ones. `apply_layout` rewrites the observable for those positions.
- `est = EstimatorV2(mode=backend)` is the Estimator for IBM backends.
- `est.options.resilience_level = 0` switches all mitigation off, and `raw = est.run([(isa, obs_isa)])` submits the first job: one circuit with one observable.
- `est.options.resilience_level = 2` switches mitigation on, and the two lines with `dd` add dynamical decoupling. `mit = est.run(...)` submits the second job.
- The last statement waits for both jobs and prints three numbers: the raw value, the mitigated value and the ideal value.

What resilience level 2 turns on, in the words of Part 1, on the tested version of qiskit-ibm-runtime:

- **Readout mitigation.** Extra circuits measure how much the readout error shrinks the result, and the result is divided by that factor. The method is called TREX (twirled readout error extinction). It has the same aim as "Undoing readout error with a matrix" but does not use the matrix.
- **Zero-noise extrapolation.** The circuit is run with its two-qubit gates folded to the noise factors 1, 3 and 5, which are the noise scales of Part 1, and the Estimator extrapolates to zero noise.
- **Twirling.** Each of these circuits is run in 32 random versions.

The raw job runs one circuit for 4,096 shots. The mitigated job runs 128 circuits and 16,384 shots. That is the "More shots and more circuits" entry of the table in Part 1, and it is the reason for the row "The mitigated run takes much longer" in the table "If something breaks" below.

What to expect with `USE_HARDWARE = False`: three numbers such as `0.8525390625 0.9729801606581271 1.0`. The raw value lands near 0.85 and the mitigated value within a few hundredths of 1. Your digits will differ, because this cell sets no seed. A mitigated value slightly above 1 is possible. No single shot has a parity above 1, but the extrapolated number is an estimate with its own scatter. The cell takes about 15 to 40 seconds on a laptop with four or more processor cores, and almost all of that is the mitigated job. With fewer cores it takes longer: on the laptop used to test this notebook, about 45 seconds with two cores and about 80 seconds with one. A hosted notebook service may give you only two cores, and slower ones. Allow a few minutes there and let the cell finish.

With `USE_HARDWARE = True` both jobs wait in the queue of the processor, the line of jobs that arrived before yours. When this lab was run on the IBM processor ibm_fez on Nov 13, 2026, the raw job used about 3 seconds of quantum time and the mitigated job about 7 seconds. Quantum time is the time a processor spends on your jobs, and it counts against what your plan allows: on the Open Plan, up to 10 minutes in every 28 days. That is one measurement. Queues and run times vary.

**Hardware cell (IBM Quantum).** With `USE_HARDWARE = True` and `HARDWARE = "ibm"` this cell uses your own IBM Quantum account. With any other setting it runs on a simulated IBM processor and uses no account.

In [ ]:
from qiskit_ibm_runtime import EstimatorV2

isa = best[3]
obs_isa = obs.apply_layout(isa.layout)
est = EstimatorV2(mode=backend)

est.options.resilience_level = 0    # raw
raw = est.run([(isa, obs_isa)])

est.options.resilience_level = 2    # mitigated
dd = est.options.dynamical_decoupling
dd.enable = True
mit = est.run([(isa, obs_isa)])

print(raw.result()[0].data.evs,
      mit.result()[0].data.evs, ideal)

**If your job is stuck in the queue.** Interrupt the kernel (Jupyter and qBraid Lab: Kernel menu, Interrupt Kernel. Google Colab: Runtime menu, Interrupt execution). Then change `USE_CACHED` to `True` in the next cell and run it. It loads the two values that were recorded on a real IBM processor, so the rest of the notebook works. Interrupting stops the waiting, not the jobs: they stay in IBM's queue and use your quantum time when they run. To take them out of the queue, run `raw.cancel()` in a new cell and `mit.cancel()` in another; a job that has finished cannot be cancelled. If your jobs finished, leave `USE_CACHED = False`. The cell then stores your own two values under the names `raw_value` and `mit_value`.

In [ ]:
# Cached result for the cell above, in case your jobs are still waiting in the queue.
# Recorded Nov 13, 2026 on the IBM processor ibm_fez.
CACHED_RAW = 0.5869
CACHED_MIT = 0.8651
USE_CACHED = False      # change to True only if your job is stuck
if USE_CACHED:
    raw_value, mit_value = CACHED_RAW, CACHED_MIT
    print("Loaded the cached result:", raw_value, mit_value, ideal)
else:
    raw_value = float(raw.result()[0].data.evs)
    mit_value = float(mit.result()[0].data.evs)
    print("Cache not loaded. The result from the cell above is kept.")

The second option can send tonight's circuit to another quantum computer. The mitigation options of the lab cell belong to Qiskit Runtime, so this route gives a raw value only. The cell below sends `qc`, the circuit before transpiling, with measurements added, and computes the parity from the counts.

It works like the second-option cells of the Session 1 notebook:

1. In the Setup section set `USE_HARDWARE = True` and `HARDWARE = "qbraid"`, then run the first two setup cells again. The first time, the setup cell installs the `qbraid` package and asks for a restart.
2. Run the cell below. Outside qBraid Lab it first asks for your qBraid API key in a hidden box. Then it prints one line per device, with the device's id, its status and its number of qubits, and stops with an error saying that the device `PASTE_DEVICE_ID` was not found. That stop is expected on the first run, because no device is chosen yet.
3. Copy the id of a device whose status reads `DeviceStatus.ONLINE` and that has at least `width` qubits, 8 unless you changed it, into `QBRAID_DEVICE` in the Setup section. Run that cell, then run the cell below again.
4. The cell waits for the job and prints the raw parity.

Two things to know about the waiting. `job.wait_for_final_state()` also returns when a job has failed, and the next line then stops with `ValueError: Counts data is not available.` A job that is put on hold (status `HOLD`) never ends the wait. In that case interrupt the kernel and ask an organizer.

**Hardware cell, second option (qBraid).** With `USE_HARDWARE = True` and `HARDWARE = "qbraid"` this cell uses your qBraid account. With any other setting it is skipped and uses no account.

In [ ]:
second_value = None                         # stays None unless the second option is switched on
provider = connect("qbraid")
if provider:
    for dev in provider.get_devices():
        print(dev.id, dev.status(), "| qubits:", dev.num_qubits)

    device = provider.get_device(QBRAID_DEVICE)
    formats = device.profile.program_spec   # the text formats this device accepts
    formats = formats if isinstance(formats, list) else [formats]
    if "qasm3" in [item.alias for item in formats]:
        device.set_target_program_type("qasm3")
    job = device.run(qc.measure_all(inplace=False), shots=1024)    # the circuit before transpiling
    job.wait_for_final_state()
    second_counts = job.result().data.get_counts()
    signed_sum = 0
    for bits, shots in second_counts.items():
        if bits.count("1") % 2 == 0:
            signed_sum += shots             # an even number of 1s counts +1
        else:
            signed_sum -= shots             # an odd number of 1s counts -1
    second_value = signed_sum / sum(second_counts.values())
    print("raw parity:", round(second_value, 3))

Step 3 of the slide is the comparison, and the slide "Run every circuit in three places" names them: the ideal simulator, the noisy simulator, and the processor. The next cell computes the ideal value with `StatevectorEstimator`, from the circuit before transpiling. It prints that value beside your two values and beside the two values that were recorded on the IBM processor ibm_fez.

In [ ]:
from qiskit.primitives import StatevectorEstimator

exact_value = float(StatevectorEstimator().run([(qc, obs)]).result()[0].data.evs)   # no noise at all
if USE_CACHED:
    mine = "cached result (same as ibm_fez below)"
elif isinstance(backend, AerSimulator):
    mine = "your run (lighter model of " + full.name + ")"
else:
    mine = "your run (" + backend.name + ")"
rows = [("ideal simulator", exact_value),
        (f"{mine}, raw", raw_value),
        (f"{mine}, mitigated", mit_value),
        ("IBM processor ibm_fez on Nov 13, 2026, raw", CACHED_RAW),
        ("IBM processor ibm_fez on Nov 13, 2026, mitigated", CACHED_MIT)]
if globals().get("second_value") is not None:
    rows.append(("second option, raw", second_value))
for label, value in rows:
    print(f"{label:50s} {value:.3f}")

Read the lines from top to bottom.

- The ideal simulator gives 1.
- Your run. On the simulated processor the raw value is near 0.85: about 15% of the parity is lost to gate errors and readout errors on eight qubits. The mitigated value is back within a few hundredths of 1.
- The IBM processor. The raw value is 0.587 and the mitigated value is 0.865. The real machine loses 41%, far more than the model, and mitigation brings back about two thirds of the loss, not all of it.

This is the third row of the "What you should see" table: the mitigated value usually sits closer to the ideal one.

Why does mitigation look better in the simulation than on the machine? The lighter model contains only the kinds of error that these methods handle best: random gate errors and readout errors on eight qubits. It has no $T_1$ or $T_2$ decay, so dynamical decoupling has nothing to act on in the simulation. The real processor has every error, including those that no calibration record lists, and there dynamical decoupling does have work to do. The simulation tells you what to expect. The processor tells you what is true.

The mitigated job keeps more than its final number. The next cell prints the value it measured at each of the three noise factors.

In [ ]:
lab_factors = None                              # the values at the three noise factors, if this run has them
if USE_CACHED:
    print("The cached result holds only the two final values.")
else:
    mit_data = mit.result()[0].data
    if hasattr(mit_data, "evs_noise_factors"):
        lab_factors = [float(value) for value in mit_data.evs_noise_factors]
        print("raw job, no mitigation:                    ", round(raw_value, 3))
        print("mitigated job at the noise factors 1, 3, 5:", [round(value, 3) for value in lab_factors])
        print("mitigated job, extrapolated to zero noise: ", round(mit_value, 3))
    else:
        print("This result does not list the values at the single noise factors.")

On the simulated processor the three values fall as the noise is raised: about 0.89, 0.76 and 0.63. The first of them is higher than the raw value, about 0.9 against about 0.85, although both were measured at the normal noise level. The difference is the readout mitigation. The step from about 0.9 to the final value near 1 is the extrapolation to zero noise. So the mitigated value is built in the two steps you met in Part 1: remove the readout error (here by rescaling, not with the matrix), then follow the trend back to zero noise. Your numbers differ a little from these and from everyone else's, because the lab code sets no seed.

#### Your turn 3

Complete the function `zne`. It takes two results, `e1` measured at the noise scale 1 and `e3` measured at the noise scale 3, and returns the zero-noise value: the point where the straight line through the two results reaches the noise scale 0. The formula is in Part 1, under "Zero-noise extrapolation in four lines".

In [ ]:
def zne(e1, e3):
    """Return the zero-noise value of a straight line through e1 (noise scale 1) and e3 (noise scale 3)."""
    e0 = None
    # YOUR CODE HERE
    return e0

In [ ]:
assert zne(1.0, 1.0) is not None, "zne() still returns None. Replace the line YOUR CODE HERE with code that sets e0."
assert isinstance(zne(1.0, 1.0), (int, float)), "zne() must return one number, the value at zero noise."
assert abs(zne(0.7, 0.7) - 0.7) < 1e-12, "If tripling the noise changes nothing, the zero-noise value is that same number."
for true_e0, slope in [(1.0, -0.1), (0.5, 0.2), (-0.3, 0.05), (1.25, -0.2)]:
    got = zne(true_e0 + slope, true_e0 + 3 * slope)
    assert abs(got - true_e0) < 1e-12, "Put a straight line through the points at the noise scales 1 and 3 and read it off at 0. Check the two weights in the formula of Part 1."
print("Check passed.")
print("Bell pair of Part 1: E(1) = 0.95 | E(3) = 0.8574 | your zero-noise value:",
      round(zne(0.95, 0.857375), 4), "| ideal: 1.0")      # 0.857375 is 0.95 ** 3, printed as 0.8574 in Part 1
factors = globals().get("lab_factors")          # the values from the cell before this exercise, if it has run
if factors is not None:
    print("Lab 3, noise factors 1 and 3:", round(factors[0], 3), "and", round(factors[1], 3),
          "| your zero-noise value:", round(zne(factors[0], factors[1]), 3),
          "| the Estimator's own value:", round(mit_value, 3))

After a passed check the cell prints two lines. The first applies your function to the Bell pair of Part 1. The second applies it to the lab circuit, with the values the mitigated job measured at the noise factors 1 and 3. If you loaded the cached result, or did not run Lab 3, the second line is missing. Your straight line and the Estimator's value need not agree. The Estimator fits a curve through all three noise factors, because, as Part 1 showed, the true trend is a curve.

## What you should see

*Slide 46.*

| Lab | Expected result |
|---|---|
| Lab 1: report card | $T_1$, $T_2$, and error rates per qubit; they differ a lot across one chip |
| Lab 2: four transpilations per seed | Different two-qubit gate counts for different levels and seeds |
| Lab 3: raw and mitigated | The mitigated value usually sits closer to the ideal one |

With `USE_HARDWARE = False` the exact outputs are these. The "Try it" cell prints `{'11': 200} {'111': 200}`. Lab 1 ends with the lines `cz (22, 23) 0.0021389143724314663` and `cz (27, 28) 1.0`. Lab 2 prints `(15, 2, 0)`. Lab 3 prints a raw value near 0.85, a mitigated value near 1, and `1.0`.

## If something breaks

*Slide 47.*

| What you see | What to do |
|---|---|
| The job stays queued | Load the cached result in the next cell and keep going |
| The backend has no calibration data | Run the cell above Lab 1 again; plain simulators carry none |
| The observable does not fit the circuit | Apply the layout to it: `obs.apply_layout(isa.layout)` |
| The mitigated run takes much longer | Expected: mitigation runs more circuits |
| `ValueError: The number of qubits of the circuit (156) does not match the number of qubits of the ()-th observable (8).` | This is the third row as Qiskit words it. Give the Estimator `obs.apply_layout(isa.layout)`, as the Lab 3 cell does |
| `IndexError: tuple index out of range` in Lab 3 or in the cell above it | Lab 1 ran after Lab 2, so `best` is a pair of qubits again. Run the Lab 2 cell again, then the small cell above the Lab 3 cell, then the Lab 3 cell |
| `ValueError: Couldn't find the property 'T1' for qubit 0.` in Lab 1 | `backend` still points at the lighter model of Lab 3. Run the small cell above the Lab 1 cell, which picks the backend again, then the Lab 1 cell |
| `AerError: 'Aer simulation failed: ERROR: a circuit requires more memory than max_memory_mb.'` in Lab 3 | The Lab 3 cell ran on the full simulated processor. Run the small cell above the Lab 3 cell, which swaps in the lighter model, then the Lab 3 cell |
| `TypeError: SamplerV2.__init__() got an unexpected keyword argument 'mode'` | The name `SamplerV2` points at Aer's class. Run the small cell after the "Try it" cell |
| `AerError: 'unknown instruction: h'` | The circuit was not transpiled for the backend. Run it through a pass manager and submit the transpiled circuit |
| `UserWarning: No positive, finite standard errors were found; falling back to an unweighted fit for extrapolation.` after Lab 3 on a real processor | Nothing to do. It is a note from the fit that takes the mitigated value back to zero noise, not an error, and the result is valid |
| IBM rejects your key (an error that mentions the API key, the token or 401) | Open the Session 1 notebook, set `REPLACE_SAVED_KEY = True` in the cell under "Save your IBM credentials once", run it with a new key, then come back here |

Notes on these rows.

- The first row. On the slide, "the next cell" is the cell after the Lab 3 cell. In this notebook that is the cached-result cell under "If your job is stuck in the queue", right after Lab 3.
- The second row. The simulated processor does carry calibration data, a saved record of ibm_fez, so Lab 1 works with `USE_HARDWARE = False`. A plain `AerSimulator()` carries none: its `properties()` method returns `None`, and the lab code then stops with `AttributeError: 'NoneType' object has no attribute 't1'`.
- The last eight rows are not on the slide. Six of them are messages that came up when the notebook was tested. The seventh is a note that a mitigated job on a real processor can print: on Nov 17, 2026 the Challenge 5 notebook, which sends the same kind of job as Lab 3, printed it on ibm_fez. The last one is for a key that IBM rejects.

## Check yourself

*Slide 39.* On the slides these questions come before the labs.

1. With $T_1 = 100\,\mu\text{s}$, how much of a 1 survives $10\,\mu\text{s}$?
2. Two qubits sit 3 steps apart. How many extra CX gates bring them together?
3. At $p = 0.1$, does the three-qubit code still help?

## Tonight in one slide

*Slide 48.*

- **Hardware.** Superconducting qubits, driven by shaped pulses.
- **Noise.** Heat, decay, readout errors and gate errors.
- **Transpile.** Fewer two-qubit gates, better answers.
- **Mitigate.** More runs, less bias in the result.
- **Correct.** Spread one qubit across many.

Steps 3 and 4 work on the processors you can use today. Step 5 is where the field is going. Tonight's challenge is steps 3 and 4 in one task.

## Challenge 5: Transpile golf

*Slide 49.*

Fit a given circuit onto an IBM processor's layout with the fewest two-qubit gates, then run it there with error mitigation. The challenge opens Fri, Nov 20 at 6:00 PM and closes Sun, Nov 22 at 11:59 PM, Clemson time (US Eastern).

The challenge notebook is posted on the event page when the challenge opens, under Challenges, "Transpile golf". Submission steps are in the challenge notebook.

## Where to go from here

*Slides 50 and 51.*

- **This weekend.** Every challenge stays open until Sun, Nov 22 at 11:59 PM. A line counts only if its form has it by then, so leave time for the queue.
- **Keep going.** Your IBM Quantum account and this week's notebooks are yours to keep using.
- **Stuck?** Ask in Discord until the challenges close.

Three things to try with this notebook. In the glue cell of Lab 2, change `width = 8` to 6 or 10 (keep it even: for an odd number of qubits the ideal value of the parity is 0, not 1), then run that cell, the Lab 2 cell, the small cell above the Lab 3 cell and the Lab 3 cell: the gate counts and the raw value change with the size of the star. In the "Try it" cell, add a second error, the line `qc.x(d[2])`, below the first one. With two flips the syndrome points to the wrong qubit, and the data reads `000` where `111` was stored, as the majority vote predicts. And if you have an IBM Quantum account, run Lab 1 with `USE_HARDWARE = True` and compare today's calibration data with the saved record.

Five sessions down. You can now write a circuit, run it on hardware, read its errors, and reason about fixing them.

- All five challenges close Sun, Nov 22 at 11:59 PM.
- Final standings and prizes for the top 5 local participants: Tue, Nov 24, 5:00 PM, McAdams 107, and in Discord.
- It is a club meeting, and anyone can walk in.

**Feedback.** Tell us how the week went: [forms.gle/ASYeymq9D3e21Kes8](https://forms.gle/ASYeymq9D3e21Kes8).